# ⛰️ 15 · Gradients, Jacobians, optimization and dynamics
### Gradient descent in one and many variables, partial derivatives and gradients, the neural-network picture, gradient identities, least squares and the normal equations, the Jacobian and the multivariable chain rule, Gauss–Newton fitting, and dynamical systems — the tRNA model, equilibria and stability from eigenvectors, and the linearised SIR epidemic

**BiologyToMath** · Part D — Calculus · Module 15 of 28

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/math/tutorial/BiologyToMath/notebooks/15_multivariable_calculus_dynamics.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, SymPy (gradients, Jacobians, Hessians, eigenvectors), SciPy (`solve_ivp`, `optimize`), pandas, Matplotlib (incl. 3-D surfaces) |
| **Prerequisites** | [Module 14 · 📈 Calculus: derivatives, integrals and rates of change](14_calculus_derivatives_integrals.ipynb) (derivatives, the chain rule, differential equations, optimisation in one variable); [Module 07 · ➡️ Vectors: profiles, lengths and angles](07_vectors.ipynb) and [Module 08 · 🔢 Matrices: tables you can multiply](08_matrices.ipynb) (dot products, norms, matrix products, transposes); [Module 09 · 🧮 Linear systems and matrix inverses](09_linear_systems_inverses.ipynb) (inverses, solving $A\mathbf x = \mathbf b$); [Module 11 · 🧱 Linear independence, span, basis and rank](11_independence_basis_rank.ipynb) (span, column space, rank); [Module 12 · 🌀 Eigenvalues and eigenvectors](12_eigenvalues_eigenvectors.ipynb) (eigenvalues and eigenvectors) |
| **Estimated time** | 180 min (reading + running + the ✏️ exercises) |
| **02-680 topics covered** | Application of calculus to gradient-based optimisation (objective function, $\operatorname{argmax}$, the linear approximation $f(x+\varepsilon) \approx f(x) + \varepsilon f'(x)$, critical/stationary points, the update $x - \alpha f'(x)$, stable vs unstable critical points of $\pm x^2/2$, gradient descent); neural networks, their cost function and descent along $-\eta\nabla c$; vector functions: partial derivatives and the gradient (examples $\mathbf x^\top\mathbf x$ and the Euclidean norm); useful gradient identities; least-squares minimisation and the normal equations; the Jacobian (definition, the worked example, local linearisation, the multivariable chain rule, nonlinear least squares); dynamical systems in several variables: the tRNA bound/unbound model, equilibria of linear models and the $\det M \neq 0$ lemma, stability by starting on an eigenvector; applications of calculus in biology and machine learning |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis. All data are **SIMULATED** by code: the clinical insulin study comes from the toolkit's `simulate_insulin_study` (true coefficients known); the enzyme-kinetics velocities (Michaelis–Menten with known $V_{max}$ and $K_m$) and the cell-viability dose–response data are simulated in this notebook. The rates of the tRNA model and of the SIR epidemic are **illustrative** values chosen to make the dynamics easy to see, not measurements.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Implement** gradient descent $x \leftarrow x - \alpha f'(x)$, **distinguish** stable from unstable critical points, and **choose** a step size (learning rate) that converges.
2. **Compute** partial derivatives and gradients by hand, with SymPy and with finite differences, and **explain** why the gradient points uphill, perpendicular to the contour lines.
3. **Describe** how a neural network learns by stepping along $-\eta\nabla c$, and **train** a one-neuron dose–response model whose gradient comes from the chain rule.
4. **Derive** $\nabla_{\mathbf x}(\mathbf b^\top\mathbf x) = \mathbf b$ and $\nabla_{\mathbf x}(\mathbf x^\top A\mathbf x) = (A + A^\top)\mathbf x$ component by component, and **classify** critical points in two dimensions with the Hessian.
5. **Derive** the normal equations $A^\top A\mathbf x = A^\top\mathbf b$ of least squares, **interpret** them geometrically (the residual is orthogonal to the columns of $A$) and **fit** a clinical regression both in closed form and by gradient descent.
6. **Compute** Jacobians, **use** them for local linearisation and the multivariable chain rule, and **fit** Michaelis–Menten kinetics by Gauss–Newton.
7. **Formulate** the tRNA bound/unbound model in discrete and continuous time, **find** its equilibrium $\mathbf m^\ast = -M^{-1}\mathbf c$, and **explain** what happens when $\det M = 0$.
8. **Decide** stability from eigenvalues by starting on an eigenvector, **draw** phase portraits, and **linearise** a nonlinear model (the SIR epidemic) with the Jacobian to derive the threshold $R_0 = \beta/\gamma > 1$.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, toolkit, simulate the insulin study, enzyme kinetics and a dose–response experiment | 5 min |
| 1 · Gradient descent in one variable | $\pm x^2/2$: stable and unstable critical points; learning rates; local vs global minima | 20 min |
| 2 · Partial derivatives and the gradient | $\nabla(\mathbf x^\top\mathbf x) = 2\mathbf x$, $\nabla\lVert\mathbf x\rVert$, contour maps, steepest ascent | 20 min |
| 3 · Gradient descent in many variables | the neural-network picture, choosing $\Delta\mathbf v$, a one-neuron dose–response model, the Hessian | 25 min |
| 4 · Gradient identities | $\nabla(\mathbf b^\top\mathbf x)$ and $\nabla(\mathbf x^\top A\mathbf x)$, component by component | 10 min |
| 5 · Least squares | no exact solution → minimise $\lVert A\mathbf x - \mathbf b\rVert^2$; normal equations; projection; insulin study | 25 min |
| 6 · The Jacobian | definition, the worked example, linearisation, the chain rule (back-propagation) | 20 min |
| 7 · Nonlinear least squares | Gauss–Newton for Michaelis–Menten; why not linearise the data | 15 min |
| 8 · Dynamical systems: the tRNA model | discrete and continuous models, equilibria, the $\det M$ lemma | 20 min |
| 9 · Stability | start on an eigenvector; phase portraits; discrete-time stability | 15 min |
| 10 · Nonlinear dynamics | linearise the SIR epidemic with the Jacobian; $R_0$ | 10 min |
| Wrap-up | takeaways, checkpoint, questions, lab note | 10 min |

## 🧫 The biological question

Module 14 handled functions of **one** variable. Biology is rarely that simple:

* A clinical study measures fasting **insulin** together with BMI, age and glucose in 120 adults. Which combination
  $\beta_0 + \beta_1\,\text{BMI} + \beta_2\,\text{age} + \beta_3\,\text{glucose}$ predicts insulin best? Four unknowns at
  once — and 120 equations that cannot all hold exactly.
* A biochemist measures the velocity of an enzyme at eight substrate concentrations and wants its $V_{max}$ and $K_m$.
  The model is *nonlinear* in $K_m$.
* A biologist wants a model to learn a **dose–response curve**, like the simplest possible neural network.
* Every cell keeps a pool of **transfer RNAs**. Aminoacyl-tRNA synthetases *charge* them with amino acids, ribosomes
  *use* the charged ones and release them uncharged, new tRNAs are transcribed and old ones degraded. At what levels does
  the pool settle — and if starvation knocks it out of balance, does it come back? An epidemiologist asks the same
  question of an infection introduced into a population: does it die out or take off?

All four need **derivatives of functions of several variables**: the *gradient* (which way to change many parameters to
improve a fit), the *Jacobian* (how several outputs respond to several inputs) and — for the dynamics — the
**eigenvalues** of Module 12, which decide whether a perturbed system returns to equilibrium.

| Biological question | Mathematical tool | Where |
|---|---|---|
| How does an algorithm "walk downhill" to the best parameter? When does it fail? | gradient descent in 1-D; stable/unstable critical points | §1 |
| Which direction improves a fit fastest when there are many parameters? | partial derivatives, gradient, contour maps | §2 |
| How does a neural network learn? | gradient descent on a cost; the chain rule | §3 |
| Which coefficients predict insulin best from BMI, age, glucose? | least squares, normal equations, gradient identities | §4–5 |
| How do several outputs respond to several inputs? How do errors flow back through a network? | the Jacobian, the multivariable chain rule | §6 |
| What are $V_{max}$ and $K_m$ of my enzyme? | Gauss–Newton nonlinear least squares | §7 |
| At what levels does the tRNA pool settle? | equilibria $\mathbf m^\ast = -M^{-1}\mathbf c$, $\det M$ | §8 |
| Does the pool recover after a disturbance? Does an epidemic take off? | eigenvalues, linearisation with the Jacobian, $R_0$ | §9–10 |

> 🧪 **Simulated data, with a known truth.** The insulin study comes from the course simulator with known true
> coefficients; the enzyme data come from Michaelis–Menten kinetics with $V_{max} = 12$ µmol/min/mg and $K_m = 3$ mM plus
> measurement noise; the viability data from a Hill-type dose–response curve with IC50 = 2 µM. We can therefore check
> every fitted number against the truth.

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this module needs only a CPU.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("sympy", "sympy")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sympy as sp
from scipy import integrate, optimize

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py). They generate the SIMULATED clinical insulin study (with its true coefficients and noise level) and draw 2-D vectors as arrows (used for eigenvectors in Section 9). Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def plot_vectors(vectors: list, labels=None, colors=None, ax=None, lim=None, title: str | None = None,
                 origin=(0.0, 0.0)):
    """Draw 2-D vectors as arrows starting at ``origin`` (default: the origin of the plane)."""
    if ax is None:
        _, ax = plt.subplots(figsize=(4.0, 4.0))
    default = ["tab:blue", "tab:orange", "tab:green", "tab:red", "tab:purple", "tab:brown"]
    colors = colors or [default[i % len(default)] for i in range(len(vectors))]
    ox, oy = origin
    for i, v in enumerate(vectors):
        ax.quiver(ox, oy, v[0], v[1], angles="xy", scale_units="xy", scale=1, color=colors[i], width=0.012)
        if labels:
            ax.text(ox + v[0] * 1.07, oy + v[1] * 1.07, labels[i], color=colors[i], fontsize=10,
                    ha="center", va="center")
    if lim is None:
        m = max([abs(c) for v in vectors for c in (ox + v[0], oy + v[1])] + [1.0]) * 1.25
        lim = (-m, m)
    ax.set_xlim(*lim)
    ax.set_ylim(*lim)
    ax.axhline(0, color="0.6", lw=0.8)
    ax.axvline(0, color="0.6", lw=0.8)
    ax.set_aspect("equal")
    ax.grid(alpha=0.3)
    if title:
        ax.set_title(title)
    return ax


INSULIN_TRUE_COEFFICIENTS = {"intercept": -4.0, "bmi": 0.55, "age": 0.04, "glucose": 0.06}


INSULIN_TRUE_SIGMA = 2.5


def simulate_insulin_study(n: int = 120, seed: int = 1921) -> pd.DataFrame:
    """SIMULATED clinical study: fasting insulin (µU/mL) of ``n`` adults as a linear function of
    BMI (kg/m²), age (years) and fasting glucose (mg/dL) plus Gaussian noise.  The true coefficients
    are ``INSULIN_TRUE_COEFFICIENTS`` and the noise SD is ``INSULIN_TRUE_SIGMA``.  Columns: bmi, age,
    glucose, insulin."""
    rng = np.random.default_rng(seed)
    bmi = rng.normal(27.0, 4.5, n).clip(17, 45)
    age = rng.uniform(20, 75, n)
    glucose = (88 + 1.1 * (bmi - 27) + 0.15 * (age - 45) + rng.normal(0, 9, n)).clip(65, 160)
    c = INSULIN_TRUE_COEFFICIENTS
    insulin = c["intercept"] + c["bmi"] * bmi + c["age"] * age + c["glucose"] * glucose \
        + rng.normal(0.0, INSULIN_TRUE_SIGMA, n)
    return pd.DataFrame({"bmi": bmi.round(1), "age": age.round(0), "glucose": glucose.round(0),
                         "insulin": insulin.round(2)})

print("🧰 toolkit loaded:", ", ".join(sorted(['INSULIN_TRUE_COEFFICIENTS', 'INSULIN_TRUE_SIGMA', 'plot_vectors', 'simulate_insulin_study'])))

In [ ]:
# ── 0.1 Generate this module's data (SIMULATED, with a known truth) ──
insulin = simulate_insulin_study()                                   # 120 adults: bmi, age, glucose, insulin
print("insulin study:", insulin.shape, "| true coefficients", INSULIN_TRUE_COEFFICIENTS, "| noise SD", INSULIN_TRUE_SIGMA)
print(insulin.head(3).to_string())

# Enzyme kinetics: initial velocity v (µmol/min per mg enzyme) at 8 substrate concentrations, in duplicate
MM_TRUE = {"Vmax": 12.0, "Km": 3.0}
S_conc = np.repeat([0.25, 0.5, 1, 2, 4, 8, 16, 32.0], 2)             # mM
rng_mm = np.random.default_rng(SEED + 6)
v_obs = MM_TRUE["Vmax"] * S_conc / (MM_TRUE["Km"] + S_conc) + rng_mm.normal(0, 0.3, S_conc.size)
print(f"\nenzyme kinetics: {S_conc.size} velocities, from {v_obs.min():.2f} to {v_obs.max():.2f} µmol/min/mg")

# Dose–response: fraction of viable cells after 48 h at 9 doses (log10 µM from −2 to 2), in triplicate
def logistic(z):
    """The logistic (sigmoid) function 1/(1 + e^(−z)), written with tanh so that it never overflows."""
    return 0.5 * (1 + np.tanh(0.5 * z))


HILL, IC50 = 1.5, 2.0                                                # true Hill slope and IC50 (µM)
W_TRUE, B_TRUE = -HILL * np.log(10), HILL * np.log(10) * np.log10(IC50)
log_dose = np.repeat(np.linspace(-2, 2, 9), 3)
rng_dr = np.random.default_rng(SEED + 15)
viability = np.clip(logistic(W_TRUE * log_dose + B_TRUE) + rng_dr.normal(0, 0.04, log_dose.size), 0, 1)
print(f"dose–response: {log_dose.size} wells; true weight w = {W_TRUE:.3f}, bias b = {B_TRUE:.3f}")

🔍 **What just happened?** We generated three SIMULATED data sets: the toolkit's insulin study (120 adults; true
coefficients intercept $-4$, BMI 0.55, age 0.04, glucose 0.06 and noise SD 2.5 µU/mL), sixteen enzyme velocities from
Michaelis–Menten kinetics with $V_{max} = 12$ and $K_m = 3$ mM, and 27 viability measurements of a drug with
IC50 = 2 µM. The viability curve is written as $\sigma(wx + b)$ with $x = \log_{10}$ dose: a logistic function of a
linear score — precisely a single artificial **neuron** (Section 3), with true weight $w = -3.454$ and bias
$b = 1.040$.

---
## 1 · Gradient descent in one variable

> 📍 **Where we are.** Module 14 ended with optimisation in one variable: write the objective $f$, solve $f'(x) = 0$,
> classify with $f''$. That recipe needs us to *solve* $f'(x) = 0$, which is impossible by hand for most models of
> interest. Here we turn the derivative into an **algorithm** that walks downhill by itself — the algorithm that trains
> nearly every machine-learning model — and we meet the idea of **stability** that returns in Sections 8–10.

**Optimisation** is the task of minimising or maximising an **objective function** $f(x)$:
$x^\ast = \operatorname{argmax}_{x} f(x)$, and maximising $f$ is equivalent to minimising $g(x) = -f(x)$. The derivative
is the rate of change (slope), so it tells us how to change $x$. Recall the **linear approximation**:

$$\frac{df}{dx} \approx \frac{f(x+\varepsilon) - f(x)}{\varepsilon} \;\Longrightarrow\; f(x+\varepsilon) \approx f(x) + \varepsilon f'(x)
\;\Longrightarrow\; f(x+\varepsilon) - f(x) \approx \varepsilon f'(x).$$

*In words:* a small step $\varepsilon$ changes $f$ by about $\varepsilon$ times the slope, so **we can reduce $f$ by moving
in small steps towards the opposite sign of the derivative**. When $f'(x) = 0$ we can change $x$ and (to first order)
nothing happens: a **critical** or **stationary point**.

**Example 1 — minimise $f(x) = \frac{x^2}{2}$**, so $f'(x) = x$. If $x > 0$, $f'(x) > 0$, and to reach the critical point we
add a *negative* $\varepsilon$ (we want to minimise). If $x < 0$, $f'(x) < 0$ and we add a *positive* $\varepsilon$. If
$x = 0$, $f'(x) = 0$: we have found a critical point.

**Example 2 — still look for the minimum, but of $g(x) = \frac{-x^2}{2}$**, so $g'(x) = -x$. It is still the case that we
add an $\varepsilon$ *opposite to the sign of the slope*: if $g'(x) < 0$ then $\varepsilon > 0$; if $g'(x) > 0$ then
$\varepsilon < 0$; and $g'(x) = 0$ is a critical point. Making the step proportional to the slope gives the rule

$$x_{\text{new}} = x - \alpha\,g'(x),$$

*in words:* step against the slope, by an amount proportional to the slope; $\alpha > 0$ is an adjustable parameter
called the **step size** or **learning rate**.

In [ ]:
# ── 1.1 The same update rule on f = x²/2 and g = −x²/2: one critical point is stable, the other unstable ──
def descend(deriv, x0, alpha, n_steps):
    """Gradient descent in one variable: x ← x − α·f'(x), repeated n_steps times. Returns all iterates."""
    xs = [x0]
    for _ in range(n_steps):
        xs.append(xs[-1] - alpha * deriv(xs[-1]))
    return np.array(xs)


f_bowl, df_bowl = (lambda x: x ** 2 / 2), (lambda x: x)            # f(x) = x²/2,  f'(x) = x
g_cap, dg_cap = (lambda x: -x ** 2 / 2), (lambda x: -x)            # g(x) = −x²/2, g'(x) = −x
ALPHA = 0.3
for x0 in [0.5, -0.1, 0.0]:
    xf, xg = descend(df_bowl, x0, ALPHA, 10), descend(dg_cap, x0, ALPHA, 10)
    print(f"start x0 = {x0:+.1f}:  on f after 10 steps x = {xf[-1]:+.5f}   |   on g after 10 steps x = {xg[-1]:+.4f}")
print(f"each step multiplies x by (1 − α) = {1 - ALPHA} on f and by (1 + α) = {1 + ALPHA} on g")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
xx = np.linspace(-3, 3, 200)
for ax, fun, der, name, col in [(axes[0], f_bowl, df_bowl, "f(x) = x²/2  (stable)", "tab:blue"),
                                (axes[1], g_cap, dg_cap, "g(x) = −x²/2  (unstable)", "tab:red")]:
    ax.plot(xx, fun(xx), color=col)
    starts = [(2.5, "o"), (-2.0, "s")] if fun is f_bowl else [(0.3, "o"), (-0.15, "s")]
    for x0, mk in starts:
        path = descend(der, x0, ALPHA, 8)
        path = path[np.abs(path) <= 3]
        ax.plot(path, fun(path), mk + "--", ms=4, lw=0.8, color="k")
        ax.plot(path[0], fun(path[0]), mk, ms=9, mfc="none", mec="k")       # open marker = start
    ax.plot(0, 0, "*", color="gold", ms=14, mec="k"); ax.set_title(name); ax.set_xlabel("x"); ax.set_ylim(-4.6, 4.6)
plt.tight_layout(); plt.show()

🔍 **What just happened?** On $f = x^2/2$ every start moves into the critical point (open markers = starts): from $x_0 = 0.5$ the iterate is
$0.5 \cdot 0.7^{10} = 0.01412$ after ten steps, because each step multiplies $x$ by $1 - \alpha = 0.7$. On $g = -x^2/2$ the
*same rule* pushes the iterate **away** from 0: each step multiplies $x$ by $1.3$, so $0.5$ becomes $6.89$ and even a start
of $-0.1$ drifts to $-1.38$. Only a start *exactly* at 0 stays there. So $x = 0$ is a critical point of both functions,
but if we were slightly away from 0, the derivative of $f$ would send us **towards** 0 and that of $g$ **away** from it:
the critical point is **stable** for $f(x)$ and **unstable** for $g(x)$. (For $g$, 0 is a maximum: descending from it
makes sense — the function has no minimum at all.)

What we have described is a slight simplification of **gradient descent**, first described by Augustin-Louis Cauchy
in 1847 and today one of the most commonly used optimisation procedures in machine learning.

⚠️ **Common confusion.** You may see the date of Cauchy's method given as 1867; his note *Méthode générale pour la
résolution des systèmes d'équations simultanées* appeared in 1847.

In [ ]:
# ── 1.2 The learning rate: too small, just right, oscillating, diverging (f = x²/2 from x0 = 2) ──
rates = {0.1: "tab:blue", 0.5: "tab:green", 1.0: "k", 1.5: "tab:orange", 2.0: "tab:purple", 2.1: "tab:red"}
for alpha, col in rates.items():
    path = descend(df_bowl, 2.0, alpha, 30)
    print(f"α = {alpha:3.1f}: factor (1 − α) = {1 - alpha:+.1f} → x after 3 steps {path[3]:+8.4f}, after 30 steps {path[-1]:+.3e}")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
for alpha, col in rates.items():
    path = descend(df_bowl, 2.0, alpha, 30)
    axes[0].plot(path[:12], "o-", ms=3, color=col, label=f"α = {alpha}")
    axes[1].semilogy(np.abs(path) + 1e-17, "o-", ms=2, color=col, label=f"α = {alpha}")
axes[0].set_xlabel("step t"); axes[0].set_ylabel("x_t"); axes[0].set_ylim(-4, 4); axes[0].set_title("first 11 steps")
axes[1].set_xlabel("step t"); axes[1].set_ylabel("|x_t| (log scale)"); axes[1].set_title("converge iff |1 − α| < 1")
axes[1].legend(fontsize=7, ncol=2); plt.tight_layout(); plt.show()

🔍 **What just happened?** For $f = x^2/2$ the update is $x_{t+1} = x_t - \alpha x_t = (1 - \alpha)x_t$, so after $t$ steps
$x_t = (1 - \alpha)^t x_0$. With $\alpha = 0.1$ the factor is 0.9 and the walk creeps (still 0.085 after 30 steps); with
$\alpha = 0.5$ it halves every step; with $\alpha = 1$ it lands on 0 in a single step (only for this function!); with
$\alpha = 1.5$ the factor $-0.5$ makes the iterate **overshoot** and flip sign every step while still converging; at
$\alpha = 2$ it bounces between $+2$ and $-2$ for ever; and at $\alpha = 2.1$ every bounce is larger — **divergence**. The
condition is $\lvert 1 - \alpha\rvert < 1$, i.e. $0 < \alpha < 2$; in general the safe learning rate is set by the curvature
$f''$ ($\alpha < 2/f''$). This is the same condition as for Euler's method on $\frac{dy}{dt} = -ky$ in Module 14 — gradient
descent *is* Euler's method applied to "move downhill at a speed equal to the slope".

Real objectives have several valleys. The next example plots a quartic with arrows showing the direction of
descent from several points: whichever valley you start in, you end up in its bottom — not necessarily the deepest one.

In [ ]:
# ── 1.3 An example with two valleys: f(x) = x⁴ + 7x³ + 5x² − 17x + 3 ──
quartic = np.poly1d([1, 7, 5, -17, 3])
d_quartic = quartic.deriv()
crit = np.sort(d_quartic.roots.real)
for xc in crit:
    kind = "minimum" if quartic.deriv(2)(xc) > 0 else "maximum"
    print(f"critical point x = {xc:+.4f}: f = {quartic(xc):+8.3f}, f'' = {quartic.deriv(2)(xc):+7.2f} → local {kind}")
for x0 in [-5.8, -2.5, -1.0, 1.8]:
    end = descend(d_quartic, x0, 0.005, 2000)[-1]
    print(f"gradient descent (α = 0.005) from x0 = {x0:+.1f} → x = {end:+.4f}, f = {quartic(end):+.3f}")

xx = np.linspace(-6, 2, 400)
fig, ax = plt.subplots(figsize=(7.2, 4.0))
ax.plot(xx, quartic(xx), color="tab:red", lw=2.5)
for x0 in [-5.8, -2.5, -1.0, 1.8]:                                  # arrows: where one big descent step would go
    x1 = x0 - 0.012 * d_quartic(x0) if abs(d_quartic(x0)) < 60 else x0 - np.sign(d_quartic(x0)) * 0.6
    ax.annotate("", xy=(x1, quartic(x1) - 3), xytext=(x0, quartic(x0) + 3),
                arrowprops=dict(arrowstyle="-|>", color="k", lw=1.5))
xg = crit[0]
ax.plot([-6, xg], [quartic(xg)] * 2, "b--", lw=2); ax.plot([xg, xg], [-60, quartic(xg)], "b--", lw=2)
ax.text(-4.0, 40, "x⁴ + 7x³ + 5x² − 17x + 3", fontsize=10)
ax.set_xlim(-6, 2); ax.set_ylim(-60, 70); ax.grid(alpha=0.4)
ax.set_xlabel("value of parameter"); ax.set_ylabel("objective"); ax.set_title("descending from four starting points")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The quartic has two valleys: a **global** minimum at $x = -4.480$ ($f = -47.07$, marked by the
blue dashed lines) and a **local** minimum at $x = 0.662$ ($f = -3.84$), separated by a local maximum at $x = -1.432$
($f = 21.25$). Gradient descent from $-5.8$ or $-2.5$ (left of the hump) reaches the global minimum; from $-1.0$ or $1.8$
it reaches the local one and stops there — the slope is zero, so the algorithm has no reason to move. The arrows show
the first step direction from each start: always downhill, never "towards the best answer". In machine learning the
standard remedies are **random restarts** (try many starting points) and noise in the steps (stochastic gradient descent).

✏️ **Your turn.** (1) For $f(x) = 3x^2$ (curvature $f'' = 6$), predict the largest learning rate that still converges and
check it with `descend`. (2) Start gradient descent on the quartic at $x_0 = -1.43$ and at $x_0 = -1.44$. Explain the
result with the word *unstable*.

### 📘 Lecture notes — gradient-based optimisation in one variable

**Optimisation:** minimise or maximise an **objective function** $f(x)$; $x^\ast = \operatorname{argmax}_{\forall x}f(x)$
(the location of the maximum); maximising $f$ ≡ minimising $g(x) = -f(x)$.

**Linear approximation:** $\frac{df}{dx} \approx \frac{f(x+\varepsilon) - f(x)}{\varepsilon}$, so
$f(x+\varepsilon) \approx f(x) + \varepsilon f'(x)$ and $f(x+\varepsilon) - f(x) \approx \varepsilon f'(x)$: we can reduce
$f$ by moving in small steps towards the opposite sign of the derivative. **Critical / stationary point:** $f'(x) = 0$
(we can change $x$, but nothing happens).

**Gradient descent:** $x \leftarrow x - \alpha f'(x)$ with step size (learning rate) $\alpha > 0$; introduced by Cauchy
(1847).

**Worked examples.** $f(x) = x^2/2$, $f'(x) = x$: $x_{t+1} = (1-\alpha)x_t$ → the critical point 0 is **stable** (nearby
starts are drawn in). $g(x) = -x^2/2$, $g'(x) = -x$: $x_{t+1} = (1+\alpha)x_t$ → 0 is **unstable** (nearby starts are
pushed away). For $x^2/2$: convergence iff $0 < \alpha < 2$ (oscillating for $1 < \alpha < 2$). The quartic
$x^4 + 7x^3 + 5x^2 - 17x + 3$ has a global minimum at $x \approx -4.48$ ($f \approx -47.1$) and a local one at
$x \approx 0.66$; descent finds whichever valley it starts in.

> 🎓 **Exam tip.** Write the update with the right sign (descent: minus), compute the factor $1 - \alpha f''$ for a
> quadratic to decide convergence, and remember that gradient descent stops at *any* critical point it reaches — local
> minima (and, if started exactly there, maxima) included.

---
## 2 · Functions of several variables: partial derivatives and the gradient

> 📍 **Where we are.** A regression has several coefficients, a neural network millions of weights. When a function
> has several input variables, $f:\mathbb R^n \to \mathbb R$, we cannot simply take "the derivative of the whole thing":
> changing $x_1$ and changing $x_2$ have different effects. We take one derivative **per input** and collect them in a
> vector, the **gradient**. Vectors and norms come from Module 07.

**Partial derivative.** For $f(\mathbf x)$ with $\mathbf x = (x_1, x_2, \dots, x_n)$, the partial derivative with respect
to $x_i$ is

$$\frac{\partial f(\mathbf x)}{\partial x_i} = \lim_{h\to0}\frac{f(x_1, \dots, x_i + h, \dots, x_n) - f(\mathbf x)}{h} .$$

*In words:* nudge only the $i$-th input, hold all others fixed, and take the ordinary derivative of Module 14. The
curly $\partial$ ("partial", "del") signals that other variables exist and are being held constant.

**Gradient.** Collect all $n$ partial derivatives:

$$\nabla f = \frac{df}{d\mathbf x} = \begin{bmatrix}\dfrac{\partial f}{\partial x_1} & \dfrac{\partial f}{\partial x_2} & \cdots & \dfrac{\partial f}{\partial x_n}\end{bmatrix}
\qquad\text{or, as a column,}\qquad \nabla_{\mathbf x}f(\mathbf x) \overset{\text{def}}{=} \begin{pmatrix}\partial f/\partial x_1\\ \vdots\\ \partial f/\partial x_n\end{pmatrix} \in \mathbb R^n .$$

*In words:* the gradient (read "nabla f" or "grad f") is "sort of $\frac{df}{dx}$ when $x$ is a vector" — a list of $n$
slopes, one per input direction. **Its dimension equals the dimension of the input** of $f$ (not of the output, which
is a single number).

⚠️ **Common confusion — row or column?** Some texts write the gradient as a **row** vector (the convention that makes
it the $1\times n$ Jacobian of Section 6), others as a **column** vector in $\mathbb R^n$ (the convention that lets us
write the update $\mathbf x - \eta\nabla f$ and identities like $\nabla(\mathbf b^\top\mathbf x) = \mathbf b$). Both
contain the same numbers. In this course we use the **column** convention for gradients and the row convention only
inside Jacobians. Also, some printed formulas use $\delta$ instead of $\partial$ for partial derivatives — it means the
same thing here.

**Example 1.** Let $f:\mathbb R^n\to\mathbb R$, $f(\mathbf x) = \mathbf x^\top\mathbf x = x_1^2 + x_2^2 + \cdots + x_n^2$. What
is $\nabla_{\mathbf x}f(\mathbf x)$? Only one term contains $x_i$, so $\frac{\partial f}{\partial x_i} = 2x_i$ and

$$\nabla_{\mathbf x}(\mathbf x^\top\mathbf x) = \begin{pmatrix}2x_1\\ \vdots\\ 2x_n\end{pmatrix} = 2\mathbf x .$$

*In words:* the squared length of a vector grows fastest if you lengthen the vector in its own direction, at a rate
twice its length — the multivariable version of $\frac{d}{dx}x^2 = 2x$.

**Example 2 — the Euclidean norm** $e(\mathbf x) = \sqrt{x_1^2 + x_2^2}$ (the length $\lVert\mathbf x\rVert$, Module 07).
First the chain rule (outside $\sqrt u = u^{1/2}$, inside $u = x_1^2 + x_2^2$), then the polynomial rule for both
derivatives:

$$\frac{\partial e}{\partial x_1} = \frac12\big(x_1^2 + x_2^2\big)^{-1/2}\cdot 2x_1 = \frac{x_1}{\sqrt{x_1^2 + x_2^2}},\qquad
\frac{\partial e}{\partial x_2} = \frac{x_2}{\sqrt{x_1^2 + x_2^2}}, \qquad
\nabla e = \frac{\mathbf x}{\lVert\mathbf x\rVert} .$$

*In words:* the gradient of the length is the **unit vector** pointing straight away from the origin: moving outward is
the fastest way to get longer, at rate exactly 1. (At $\mathbf x = \mathbf 0$ the norm has a sharp tip like $\lvert x\rvert$
and no gradient.) When you copy this, note that the second entry is $\partial e/\partial x_2$ — not a second
$\partial e/\partial x_1$.

In [ ]:
# ── 2.1 Gradients symbolically (SymPy) and numerically (central differences, one coordinate at a time) ──
def numerical_gradient(f, x, h=1e-6):
    """Approximate ∇f(x) by central differences: nudge one coordinate at a time by ±h."""
    x = np.asarray(x, dtype=float)
    grad = np.zeros_like(x)
    for i in range(x.size):
        step = np.zeros_like(x)
        step[i] = h
        grad[i] = (f(x + step) - f(x - step)) / (2 * h)
    return grad


x1, x2, x3 = sp.symbols("x1 x2 x3", real=True)
X3 = sp.Matrix([x1, x2, x3])
f_sq = (X3.T * X3)[0]                                                 # xᵀx for n = 3
print("f = xᵀx =", f_sq, "→ ∇f =", [sp.diff(f_sq, v) for v in (x1, x2, x3)])
e_norm = sp.sqrt(x1 ** 2 + x2 ** 2)
grad_e = [sp.simplify(sp.diff(e_norm, v)) for v in (x1, x2)]
print("e = √(x1² + x2²) → ∇e =", grad_e)

pt = np.array([3.0, 4.0])
norm2 = lambda v: np.sqrt(v @ v)
print(f"at x = (3, 4): ∇e formula = {pt / norm2(pt)}, numerical = {numerical_gradient(norm2, pt).round(8)}, "
      f"length {norm2(pt / norm2(pt)):.1f}")
pt3 = np.array([1.0, -2.0, 0.5])
print(f"at x = (1, −2, 0.5): ∇(xᵀx) formula 2x = {2 * pt3}, numerical = {numerical_gradient(lambda v: v @ v, pt3).round(8)}")

🔍 **What just happened?** SymPy confirms $\nabla(\mathbf x^\top\mathbf x) = (2x_1, 2x_2, 2x_3)$ and
$\nabla e = \big(x_1/\sqrt{x_1^2 + x_2^2},\; x_2/\sqrt{x_1^2 + x_2^2}\big)$. At $\mathbf x = (3, 4)$ (length 5) the gradient of
the norm is $(0.6, 0.8)$ — the unit vector in the direction of $\mathbf x$ — and the finite-difference gradient agrees to
eight decimals; at $(1, -2, 0.5)$ the gradient of $\mathbf x^\top\mathbf x$ is $2\mathbf x = (2, -4, 1)$. The function
`numerical_gradient` (one central difference per coordinate, Module 14) is our all-purpose checker for the rest of the
module.

**Why the gradient points uphill.** Move from $\mathbf x$ a small distance $\varepsilon$ in the direction of a unit
vector $\mathbf u$. The linear approximation in several variables (one term per input, each "slope × step") is

$$f(\mathbf x + \varepsilon\mathbf u) \approx f(\mathbf x) + \varepsilon\left(\frac{\partial f}{\partial x_1}u_1 + \cdots + \frac{\partial f}{\partial x_n}u_n\right)
= f(\mathbf x) + \varepsilon\,\nabla f\cdot\mathbf u .$$

*In words:* the rate of change of $f$ in direction $\mathbf u$ — the **directional derivative** — is the dot product
$\nabla f\cdot\mathbf u = \lVert\nabla f\rVert\cos\theta$ (Module 07), where $\theta$ is the angle between $\mathbf u$ and the
gradient. It is largest ($= \lVert\nabla f\rVert$) for $\theta = 0$: the gradient points in the direction of **steepest
ascent**, and its length is the steepness. It is most negative for $\theta = 180°$: $-\nabla f$ is the direction of
**steepest descent**. And it is zero for $\theta = 90°$: moving perpendicular to the gradient keeps $f$ constant, so
**the gradient is perpendicular to the contour lines** (the curves of equal $f$, like height lines on a hiking map).

In [ ]:
# ── 2.2 Contour maps with gradient arrows: the Euclidean norm and an elongated bowl ──
def bowl(v):
    """A tilted, elongated quadratic bowl c(v1, v2) = v1² + 1.2·v1·v2 + 4·v2² (a stand-in for a cost function)."""
    return v[0] ** 2 + 1.2 * v[0] * v[1] + 4 * v[1] ** 2


def grad_bowl(v):
    """Its gradient (2v1 + 1.2v2, 1.2v1 + 8v2)."""
    return np.array([2 * v[0] + 1.2 * v[1], 1.2 * v[0] + 8 * v[1]])


G1, G2 = np.meshgrid(np.linspace(-3, 3, 200), np.linspace(-3, 3, 200))
Q1, Q2 = np.meshgrid(np.linspace(-2.5, 2.5, 9), np.linspace(-2.5, 2.5, 9))
fig, axes = plt.subplots(1, 2, figsize=(10, 4.3))
axes[0].contour(G1, G2, np.sqrt(G1 ** 2 + G2 ** 2), levels=10, cmap="viridis", linewidths=0.8)
R = np.sqrt(Q1 ** 2 + Q2 ** 2) + 1e-12
axes[0].quiver(Q1, Q2, Q1 / R, Q2 / R, color="tab:red", angles="xy", scale=12, width=0.006)
axes[0].set_title("e(x) = ‖x‖: ∇e = x/‖x‖ (unit length, radial)")
axes[1].contour(G1, G2, bowl([G1, G2]), levels=np.geomspace(0.2, 40, 12), cmap="viridis", linewidths=0.8)
gb = grad_bowl([Q1, Q2])
axes[1].quiver(Q1, Q2, gb[0], gb[1], color="tab:red", angles="xy", scale=250, width=0.006)
axes[1].set_title("bowl c(v): ∇c is perpendicular to the contours, uphill")
for ax, lab in zip(axes, [("x1", "x2"), ("v1", "v2")]):
    ax.set_aspect("equal"); ax.set_xlabel(lab[0]); ax.set_ylabel(lab[1])
plt.tight_layout(); plt.show()

point = np.array([1.5, 1.0])
angles = np.linspace(0, 2 * np.pi, 3600, endpoint=False)
units = np.c_[np.cos(angles), np.sin(angles)]
slopes = units @ grad_bowl(point)                                     # directional derivatives ∇c·u
best = units[slopes.argmax()]
print(f"at v = {point}: ∇c = {grad_bowl(point)}, ‖∇c‖ = {np.linalg.norm(grad_bowl(point)):.4f}")
print(f"steepest direction among 3600 unit vectors = {best.round(4)}, ∇c/‖∇c‖ = {(grad_bowl(point) / np.linalg.norm(grad_bowl(point))).round(4)}, "
      f"largest slope {slopes.max():.4f}")

🔍 **What just happened?** Left: the contours of $\lVert\mathbf x\rVert$ are circles and every gradient arrow has length 1
and points radially outward, perpendicular to the circles. Right: for the tilted bowl the arrows are longer where the
contours are packed together (steep) and always cross the contours at right angles, pointing uphill. Trying 3,600 unit
directions at $\mathbf v = (1.5, 1)$, where $\nabla c = (4.2, 9.8)$, the fastest increase occurs along $(0.3939, 0.9191)$ —
the normalised gradient $\nabla c/\lVert\nabla c\rVert$ — and the largest slope equals $\lVert\nabla c\rVert = 10.6621$, exactly as
the dot-product argument predicts. The steepest *descent* direction is therefore $-\nabla c$.

✏️ **Your turn.** (1) Compute by hand the gradient of $f(x_1, x_2) = x_1^2x_2 + e^{x_2}$ and check it with
`numerical_gradient` at $(1, 0)$. (2) Show that the gradient of $\lVert\mathbf x\rVert$ in $\mathbb R^n$ is
$\mathbf x/\lVert\mathbf x\rVert$ for every $n$ (write $\lVert\mathbf x\rVert = (\mathbf x^\top\mathbf x)^{1/2}$ and use Example 1
with the chain rule).

### 📘 Lecture notes — partial derivatives and the gradient

**Vector functions.** For $f(\mathbf x)$ with $\mathbf x = (x_1, \dots, x_n)$:
$\frac{\partial f(\mathbf x)}{\partial x_i} = \lim_{h\to0}\frac{f(x_1, \dots, x_i + h, \dots, x_n) - f(\mathbf x)}{h}$ (vary one
input, hold the others fixed), and the **gradient** $\nabla f = \frac{df}{d\mathbf x}$ collects them:
$\big[\frac{\partial f}{\partial x_1}\;\cdots\;\frac{\partial f}{\partial x_n}\big]$ (row form) or
$\nabla_{\mathbf x}f(\mathbf x) \in \mathbb R^n$ (column form, used here). For $f:\mathbb R^n \to \mathbb R$ the gradient has
$n$ entries — **its dimension is that of the input, not the output**.

**Examples.** $f(\mathbf x) = \mathbf x^\top\mathbf x \Rightarrow \nabla_{\mathbf x}f = 2\mathbf x$. Euclidean norm
$e(\mathbf x) = \sqrt{x_1^2 + x_2^2}$: chain rule, then polynomial rule, gives
$\nabla e = \big[\frac{\partial e}{\partial x_1}\;\frac{\partial e}{\partial x_2}\big] = \Big[\frac{x_1}{\sqrt{x_1^2+x_2^2}}\;\;\frac{x_2}{\sqrt{x_1^2+x_2^2}}\Big] = \frac{\mathbf x}{\lVert\mathbf x\rVert}$.

**Geometry.** Directional derivative $\nabla f\cdot\mathbf u = \lVert\nabla f\rVert\cos\theta$: the gradient points in the
direction of steepest ascent, its length is the steepness, $-\nabla f$ is steepest descent, and $\nabla f$ is
perpendicular to contour lines.

> 🎓 **Exam tip.** To compute a gradient, treat every other variable as a constant and differentiate with the rules of
> Module 14 — one partial derivative per input. Check the dimension (= number of inputs) and, if possible, one entry
> numerically.

---
## 3 · Gradient descent in many variables — how a neural network learns

> 📍 **Where we are.** Section 1 descended in one variable; Section 2 showed that $-\nabla f$ is the steepest way down
> in many. Combining the two gives the algorithm behind modern machine learning. Its most famous application is
> training **neural networks**, so we start with their picture.

A (feed-forward) **neural network** takes an input vector $\mathbf x$ (e.g. the expression of five genes), passes it
through layers of "neurons" — each computes a weighted sum of its inputs with **weights** $w$, adds a **bias** $b$ and
applies a non-linear function — and produces an output $a$ (e.g. a predicted drug response). To train it we define a
**cost function** (also called loss or objective) that measures how far the outputs are from the desired outputs over
the $m$ training examples:

$$c(w, b) = \frac{1}{2m}\sum_{x}\big\lVert f(x) - a\big\rVert^2 ,$$

*in words:* for each training input $x$, compare the desired output $f(x)$ (the measured truth) with the network's
output $a$ (which depends on all weights and biases), square the distance, and average over the $m$ examples; the
$\frac12$ is a convenience that cancels the 2 produced by differentiating a square. Learning = choosing all the weights
and biases $\mathbf v = (w, b)$ to minimise $c$.

In [ ]:
# ── 3.1 The picture: input layer, hidden layers, output, and the cost to minimise ──
def draw_network(ax, sizes=(5, 4, 4, 1), xs=(0.0, 1.6, 3.2, 5.6)):
    """Draw a small fully connected feed-forward network with matplotlib (nodes = circles, weights = arrows)."""
    pos = []
    for x_layer, n in zip(xs, sizes):
        ys = np.linspace(-(n - 1) / 2, (n - 1) / 2, n) * 0.9
        pos.append([(x_layer, y) for y in ys])
    for k in range(len(sizes) - 1):
        for (xa, ya) in pos[k]:
            for (xb, yb) in pos[k + 1]:
                if k == len(sizes) - 2:                                   # last layer drawn after a gap ("…")
                    xa_, xb_ = xa + 0.9, xb
                else:
                    xa_, xb_ = xa, xb
                ax.annotate("", xy=(xb_ - 0.13, yb), xytext=(xa_ + 0.13, ya),
                            arrowprops=dict(arrowstyle="-|>", color="0.2", lw=0.7, mutation_scale=7))
    for k, layer in enumerate(pos):
        for (x_, y_) in layer:
            ax.add_patch(plt.Circle((x_, y_), 0.13, color="#2b9be8", zorder=3))
    ax.add_patch(plt.Rectangle((-0.3, -2.2), 0.6, 4.4, fill=False, ec="tab:red", lw=2))
    ax.add_patch(plt.Rectangle((1.3, -1.75), 0.6, 3.5, fill=False, ec="tab:green", lw=2))
    ax.text(0, -2.55, "input [x]", color="tab:red", ha="center", fontweight="bold")
    ax.text(1.6, -2.1, "layer 1", color="tab:green", ha="center", fontweight="bold")
    ax.text(0.7, 2.2, "weight w", fontweight="bold"); ax.text(1.75, 1.95, "bias b", fontweight="bold")
    ax.text(4.1, 0.0, "…", fontsize=18, ha="center", va="center")
    ax.text(5.8, 0.0, "output [a]", color="tab:orange", fontweight="bold", va="center")
    ax.text(3.8, -1.9, "cost / loss / objective:\n" + r"$c(w,b) = \frac{1}{2m}\sum_x \|f(x) - a\|^2$", fontsize=10)
    ax.set_xlim(-0.5, 7.2); ax.set_ylim(-2.8, 2.6); ax.axis("off")


fig, ax = plt.subplots(figsize=(8.5, 4.0))
draw_network(ax)
ax.set_title("A neural network: every arrow carries a weight, every node of a layer a bias")
plt.tight_layout(); plt.show()
n_params = 5 * 4 + 4 + 4 * 4 + 4 + 4 * 1 + 1
print(f"even this tiny network (5 → 4 → 4 → 1) has {n_params} weights and biases: c is a function of {n_params} variables")

🔍 **What just happened?** The drawing shows the standard picture: five inputs (red box), a first layer of four neurons
(green box), a second hidden layer, and one output $a$. Counting arrows (weights) and nodes (biases), even this toy
network has **49 parameters** ($5\cdot4 + 4\cdot4 + 4\cdot1 = 40$ weights and $4 + 4 + 1 = 9$ biases), so its cost $c$ is a
function on $\mathbb R^{49}$ — far too many to plot or to solve
$\nabla c = \mathbf 0$ by hand. Real networks have millions. We picture the landscape with just **two** parameters
$\mathbf v = (v_1, v_2)$: the cost is a surface over the $(v_1, v_2)$ plane, and the current parameters are a point on
it.

In [ ]:
# ── 3.2 The cost as a surface: a point (v1', v2') and one step to (v1' + Δv1, v2' + Δv2) ──
V1, V2 = np.meshgrid(np.linspace(-2.6, 2.6, 60), np.linspace(-2.6, 2.6, 60))
C = bowl([V1, V2])
checker = ((np.floor(V1 / 0.26) + np.floor(V2 / 0.26)) % 2).astype(bool)
colours = np.where(checker[..., None], np.array([0.2, 0.75, 0.2, 1.0]), np.array([0.05, 0.45, 0.05, 1.0]))
v_now = np.array([-2.0, 1.2])
eta = 0.08
v_next = v_now - eta * grad_bowl(v_now)                              # one gradient-descent step
print(f"(v1', v2') = {v_now}, c = {bowl(v_now):.3f}; ∇c = {grad_bowl(v_now)}")
print(f"step Δv = −η∇c = {(-eta * grad_bowl(v_now)).round(3)} → new point {v_next.round(3)}, c = {bowl(v_next):.3f}")

fig = plt.figure(figsize=(7.5, 5))
ax = fig.add_subplot(projection="3d", computed_zorder=False)      # draw in the order given: surface first, points on top
ax.plot_surface(V1, V2, C, facecolors=colours, rstride=1, cstride=1, linewidth=0, antialiased=False, shade=False,
                alpha=0.85, zorder=1)
ax.plot([v_now[0], v_next[0]], [v_now[1], v_next[1]], [bowl(v_now) + 0.5, bowl(v_next) + 0.5], color="darkred", lw=3,
        zorder=10)
ax.scatter(*v_now, bowl(v_now) + 0.5, color="red", s=80, depthshade=False, zorder=11)
ax.scatter(*v_next, bowl(v_next) + 0.5, color="pink", s=80, edgecolor="k", depthshade=False, zorder=11)
ax.text(v_now[0] - 0.3, v_now[1], bowl(v_now) + 4, "(v1', v2')", fontweight="bold", zorder=12)
ax.text(v_next[0] + 0.2, v_next[1] - 0.6, bowl(v_next) - 7, "(v1'+Δv1, v2'+Δv2)", fontweight="bold", zorder=12)
ax.set_xlabel("v1"); ax.set_ylabel("v2"); ax.set_zlabel("cost c"); ax.view_init(elev=28, azim=-60)
ax.set_title("the cost surface over two parameters, and one downhill step"); plt.tight_layout(); plt.show()

🔍 **What just happened?** We drew the (stand-in) cost $c(v_1, v_2)$ of Section 2.2 as a checkered surface, put the
current parameters at $(v_1', v_2') = (-2, 1.2)$ where $c = 6.88$, and took one step to
$(v_1' + \Delta v_1, v_2' + \Delta v_2) = (-1.795, 0.624)$, where $c = 3.436$ — half the cost, in one step.

**How do we choose $\Delta\mathbf v$ to minimise $c$?** Use the linear approximation of Section 2 for a small step
$\Delta\mathbf v = (\Delta v_1, \Delta v_2)$:

$$\Delta c \approx \frac{\partial c}{\partial v_1}\Delta v_1 + \frac{\partial c}{\partial v_2}\Delta v_2 = \nabla c\cdot\Delta\mathbf v .$$

*In words:* the change of the cost is approximately the dot product of the gradient with the step. We want $\Delta c$ to
be negative. Choose

$$\Delta\mathbf v = -\eta\,\nabla c \quad\Longrightarrow\quad \Delta c \approx -\eta\,\nabla c\cdot\nabla c = -\eta\,\lVert\nabla c\rVert^2 \le 0 ,$$

*in words:* step against the gradient, scaled by a small positive **learning rate** $\eta$ (Greek *eta*); then the
cost decreases (by about $\eta$ times the squared length of the gradient) unless the gradient is already zero. Another
view: we take small steps in the direction that most quickly decreases $c$, which is $-\nabla c$ (Section 2). The
update rule

$$\mathbf v \leftarrow \mathbf v - \eta\,\nabla c(\mathbf v)$$

is **gradient descent** in many variables — Section 1's $x \leftarrow x - \alpha f'(x)$ with the derivative replaced by
the gradient. Check with our numbers: $\eta\lVert\nabla c\rVert^2 = 0.08 \times (2.56^2 + 7.2^2) = 4.67$, so the linear
approximation predicts a drop from 6.88 to 2.21; the actual drop (to 3.44) is smaller because the step is not
infinitesimal — the bowl curves upward.

In [ ]:
# ── 3.3 Gradient-descent paths on the bowl: the learning rate again ──
def gradient_descent(grad, v0, eta, n_steps):
    """v ← v − η ∇c(v), repeated; returns the path (one row per iterate)."""
    path = [np.asarray(v0, dtype=float)]
    for _ in range(n_steps):
        path.append(path[-1] - eta * grad(path[-1]))
    return np.array(path)


H_bowl = np.array([[2.0, 1.2], [1.2, 8.0]])                          # matrix of second derivatives of the bowl
lam = np.linalg.eigvalsh(H_bowl)
print(f"eigenvalues of the bowl's second-derivative matrix: {lam.round(3)} → stable if η < 2/{lam.max():.3f} = {2 / lam.max():.3f}")
fig, axes = plt.subplots(1, 4, figsize=(13, 3.4))
for ax, eta_ in zip(axes, [0.02, 0.1, 0.22, 0.25]):
    path = gradient_descent(grad_bowl, [-2.4, 1.6], eta_, 40)
    ax.contour(G1, G2, bowl([G1, G2]), levels=np.geomspace(0.05, 40, 14), cmap="viridis", linewidths=0.7)
    ax.plot(path[:, 0], path[:, 1], "o-", ms=2.5, lw=1, color="tab:red")
    ax.set_xlim(-3, 3); ax.set_ylim(-3, 3); ax.set_aspect("equal")
    ax.set_title(f"η = {eta_}: c after 40 steps = {bowl(path[-1]):.2g}", fontsize=9); ax.set_xlabel("v1")
    print(f"η = {eta_:4.2f}: c(v) after 10 steps {bowl(path[10]):9.4f}, after 40 steps {bowl(path[-1]):.3e}")
axes[0].set_ylabel("v2"); plt.tight_layout(); plt.show()

🔍 **What just happened?** The bowl's matrix of second derivatives has eigenvalues 1.769 and 8.231 (one gentle and one
steep direction — the long, thin, tilted valley). Exactly as in one dimension, the step size must stay below
$2/\lambda_{max} = 0.243$. With $\eta = 0.02$ the path creeps along the valley floor (cost still 0.35 after 40 steps);
with $\eta = 0.1$ it converges smoothly (cost $1.1\times10^{-6}$); with $\eta = 0.22$ it **zig-zags** across the steep direction
but still converges ($2.7\times10^{-7}$); with $\eta = 0.25$, just above the limit, the zig-zag grows and the cost explodes (459
after 40 steps). Gradient descent
in many variables inherits all the behaviour of Section 1, direction by direction: the eigenvectors of the
second-derivative matrix are the independent directions, and the largest eigenvalue sets the speed limit.

### 3.4 Training a single neuron: a dose–response curve

The smallest neural network has one input, one neuron and one output: $a(x) = \sigma(wx + b)$ with the logistic
function $\sigma(z) = \frac{1}{1 + e^{-z}}$. With $x = \log_{10}(\text{dose})$ this is the Hill dose–response curve of
pharmacology (a negative weight means "more drug, fewer viable cells"). We fit it to the 27 SIMULATED viability
measurements $f(x_j)$ by minimising the cost of Section 3,

$$c(w, b) = \frac{1}{2m}\sum_{j=1}^{m}\big(f(x_j) - a_j\big)^2, \qquad a_j = \sigma(z_j),\quad z_j = wx_j + b .$$

**The gradient by the chain rule.** The cost depends on $w$ only through $z_j$ and $a_j$:
$\frac{\partial c}{\partial w} = \sum_j \frac{\partial c}{\partial a_j}\cdot\frac{da_j}{dz_j}\cdot\frac{\partial z_j}{\partial w}$, with

$$\frac{\partial c}{\partial a_j} = \frac{1}{m}\big(a_j - f(x_j)\big), \qquad \frac{d\sigma}{dz} = \frac{e^{-z}}{(1 + e^{-z})^2} = \sigma(z)\big(1 - \sigma(z)\big), \qquad \frac{\partial z_j}{\partial w} = x_j, \quad \frac{\partial z_j}{\partial b} = 1,$$

so

$$\frac{\partial c}{\partial w} = \frac1m\sum_j\big(a_j - f(x_j)\big)\,a_j(1 - a_j)\,x_j, \qquad \frac{\partial c}{\partial b} = \frac1m\sum_j\big(a_j - f(x_j)\big)\,a_j(1 - a_j).$$

*In words:* each measurement pushes the parameters in proportion to its error $a_j - f(x_j)$, times the local slope of
the S-curve $a_j(1 - a_j)$, times the input ($x_j$ for the weight, 1 for the bias). Multiplying local derivatives from the
cost back to the parameters is exactly what **back-propagation** does in a deep network, layer after layer (Section 6).
(The derivative of $\sigma$: chain rule on $(1 + e^{-z})^{-1}$, then $\frac{e^{-z}}{(1+e^{-z})^2} = \frac{1}{1+e^{-z}}\cdot\frac{e^{-z}}{1+e^{-z}} = \sigma(1-\sigma)$.)

In [ ]:
# ── 3.4 Gradient descent for one neuron a = σ(w·x + b) on the SIMULATED dose–response data ──
def neuron_cost(v, x=log_dose, f=viability):
    """c(w, b) = 1/(2m) Σ (f(x_j) − σ(w x_j + b))²."""
    return np.mean((f - logistic(v[0] * x + v[1])) ** 2) / 2


def neuron_grad(v, x=log_dose, f=viability):
    """(∂c/∂w, ∂c/∂b) from the chain rule: mean of (a − f)·a(1 − a)·(x, 1)."""
    a = logistic(v[0] * x + v[1])
    delta = (a - f) * a * (1 - a)
    return np.array([np.mean(delta * x), np.mean(delta)])


v_test = np.array([-1.0, 0.5])
print("gradient at (w, b) = (−1, 0.5): chain rule", neuron_grad(v_test).round(8), "| finite differences",
      numerical_gradient(neuron_cost, v_test).round(8))
path_nn = gradient_descent(neuron_grad, [0.0, 0.0], eta=5.0, n_steps=3000)
w_fit, b_fit = path_nn[-1]
best = optimize.minimize(neuron_cost, [0.0, 0.0]).x
print(f"gradient descent (η = 5, 3000 steps): w = {w_fit:.4f}, b = {b_fit:.4f}, cost {neuron_cost(path_nn[-1]):.3e}, "
      f"‖∇c‖ = {np.linalg.norm(neuron_grad(path_nn[-1])):.1e}")
print(f"scipy.optimize.minimize      : w = {best[0]:.4f}, b = {best[1]:.4f}")
print(f"truth                        : w = {W_TRUE:.4f}, b = {B_TRUE:.4f}")
print(f"→ Hill slope −w/ln10 = {-w_fit / np.log(10):.3f} (true {HILL}); IC50 = 10^(−b/w) = {10 ** (-b_fit / w_fit):.2f} µM (true {IC50})")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
W, B = np.meshgrid(np.linspace(-6, 0.5, 120), np.linspace(-1.5, 3.5, 120))
Cn = np.array([[neuron_cost([w_, b_]) for w_, b_ in zip(rw, rb)] for rw, rb in zip(W, B)])
axes[0].contour(W, B, Cn, levels=np.geomspace(Cn.min() * 1.05, Cn.max(), 15), cmap="viridis", linewidths=0.8)
axes[0].plot(path_nn[::25, 0], path_nn[::25, 1], "o-", ms=2, lw=1, color="tab:red", label="gradient descent (every 25th step)")
axes[0].plot(W_TRUE, B_TRUE, "k*", ms=12, label="truth")
axes[0].set_xlabel("weight w"); axes[0].set_ylabel("bias b"); axes[0].legend(fontsize=7); axes[0].set_title("cost landscape c(w, b)")
xd = np.linspace(-2.3, 2.3, 200)
axes[1].plot(log_dose, viability, "o", ms=4, color="0.4", label="SIMULATED wells")
axes[1].plot(xd, logistic(w_fit * xd + b_fit), color="tab:red", label="trained neuron")
axes[1].plot(xd, logistic(W_TRUE * xd + B_TRUE), "k--", lw=1, label="truth")
axes[1].set_xlabel("log10 dose (µM)"); axes[1].set_ylabel("fraction viable"); axes[1].legend(fontsize=7)
axes[1].set_title("dose–response = one neuron"); plt.tight_layout(); plt.show()

🔍 **What just happened?** The chain-rule gradient agrees with the finite-difference gradient to eight decimals.
Starting from $(w, b) = (0, 0)$ — a flat curve at 50 % viability — 3,000 gradient-descent steps reach
$(w, b) = (-3.444, 1.168)$ where the gradient has vanished ($\lVert\nabla c\rVert \approx 10^{-10}$), matching SciPy's
general-purpose optimiser ($-3.446$, $1.169$). Translated back into pharmacology, the trained neuron has Hill slope 1.496 (true 1.5) and IC50 = 2.18 µM
(true 2.0) — the difference is measurement noise in 27 wells, not a failure of the optimiser. In the left panel the path
heads almost straight for the valley and then curls into the minimum; the elongated, curved contours show that the
landscape of even one neuron is not a simple round bowl.

### 3.5 The second-derivative test in two variables: the Hessian

At a critical point ($\nabla f = \mathbf 0$) the first-order term of the linear approximation vanishes, and the shape is
decided by the second derivatives, collected in the symmetric **Hessian** matrix

$$H = \begin{bmatrix} \dfrac{\partial^2 f}{\partial x_1^2} & \dfrac{\partial^2 f}{\partial x_1\partial x_2}\\[2mm] \dfrac{\partial^2 f}{\partial x_2\partial x_1} & \dfrac{\partial^2 f}{\partial x_2^2}\end{bmatrix},
\qquad f(\mathbf x^\ast + \boldsymbol\varepsilon) \approx f(\mathbf x^\ast) + \tfrac12\,\boldsymbol\varepsilon^\top H\,\boldsymbol\varepsilon .$$

*In words:* near a critical point the function looks like a quadratic bowl, dome or saddle described by $H$. Along an
eigenvector of $H$ with eigenvalue $\lambda$ the function changes like $\frac12\lambda\varepsilon^2$ (Module 12), so:
**all eigenvalues positive → local minimum; all negative → local maximum; mixed signs → saddle point** (up in one
direction, down in another — like a mountain pass); a zero eigenvalue → inconclusive. This is the multivariable version
of Module 14's second-derivative test.

In [ ]:
# ── 3.5 Minimum, maximum, saddle: Hessians and their eigenvalues ──
X1, X2 = sp.symbols("x1 x2", real=True)
CASES = {"x1² + x2²": X1 ** 2 + X2 ** 2, "−x1² − x2²": -X1 ** 2 - X2 ** 2, "x1² − x2²": X1 ** 2 - X2 ** 2,
         "bowl c(v)": X1 ** 2 + sp.Rational(6, 5) * X1 * X2 + 4 * X2 ** 2}
fig, axes = plt.subplots(1, 3, figsize=(11, 3.4))
for k, (name, expr) in enumerate(CASES.items()):
    Hs = sp.hessian(expr, (X1, X2))
    ev = sorted(float(e) for e in Hs.eigenvals(multiple=True))
    kind = "minimum" if min(ev) > 0 else "maximum" if max(ev) < 0 else "saddle"
    print(f"{name:12s}: H = {Hs.tolist()}, eigenvalues {np.round(ev, 3)} → {kind} at (0, 0)")
    if k < 3:
        F = sp.lambdify((X1, X2), expr, "numpy")
        axes[k].contour(G1, G2, F(G1, G2), levels=15, cmap="RdBu_r", linewidths=0.8)
        axes[k].plot(0, 0, "k*", ms=10); axes[k].set_aspect("equal")
        axes[k].set_title(f"{name}: {kind} (λ = {ev[0]:g}, {ev[1]:g})", fontsize=9); axes[k].set_xlabel("x1")
axes[0].set_ylabel("x2"); plt.tight_layout(); plt.show()

🔍 **What just happened?** SymPy's `hessian` gives $H = 2I$ for $x_1^2 + x_2^2$ (eigenvalues 2, 2: minimum), $-2I$ for the
dome (maximum), and $\mathrm{diag}(2, -2)$ for $x_1^2 - x_2^2$: one positive and one negative eigenvalue, a **saddle** — the
contours are hyperbolas, and $(0, 0)$ is a minimum along $x_1$ but a maximum along $x_2$. Gradient descent can stall
near saddles because the gradient is zero there too; in high-dimensional neural-network landscapes saddles are far more
common than poor local minima. The bowl's Hessian is $[[2, 1.2], [1.2, 8]]$ with eigenvalues 1.769 and 8.231 — the same
numbers that set its learning-rate limit in 3.3. One matrix describes both the shape of the valley and how fast we may
descend it.

✏️ **Your turn.** (1) Find the critical point of $f(x_1, x_2) = x_1^2 + x_1x_2 + x_2^2 - 3x_1$ by solving $\nabla f = \mathbf 0$
by hand, classify it with the Hessian, and confirm with `gradient_descent`. (2) In 3.4, start the neuron at
$(w, b) = (2, 0)$ — an *increasing* dose–response. Does gradient descent still find the decreasing curve? How many
steps does it need?

### 📘 Lecture notes — gradient descent in many variables (and neural networks)

**Neural network.** Inputs $x$ → layers of neurons (weights $w$, biases $b$) → output $a$. **Cost / loss / objective**
$c(w, b) = \frac{1}{2m}\sum_x\lVert f(x) - a\rVert^2$ over the $m$ training examples ($f(x)$ = desired output).

**Choosing the step.** For a small step $\Delta\mathbf v$, $\Delta c \approx \nabla c\cdot\Delta\mathbf v$. Taking
$\Delta\mathbf v = -\eta\nabla c$ gives $\Delta c \approx -\eta\lVert\nabla c\rVert^2 \le 0$. Equivalently: take small steps in
the direction that most decreases $c$, which is $-\eta\nabla c$. Update $\mathbf v \leftarrow \mathbf v - \eta\nabla c(\mathbf v)$.

**Learning rate.** Converges if $\eta < 2/\lambda_{max}$ of the Hessian (bowl example: $2/8.231 = 0.243$); too small →
slow; near the limit → zig-zag; above → divergence.

**One neuron** $a = \sigma(wx + b)$, $\sigma' = \sigma(1-\sigma)$:
$\partial c/\partial w = \frac1m\sum_j(a_j - f_j)a_j(1-a_j)x_j$ — the chain rule, i.e. back-propagation in miniature.
Fitted to SIMULATED viability data: Hill slope 1.50, IC50 2.18 µM (truth 1.5, 2.0 µM).

**Hessian test** at a critical point: eigenvalues all $> 0$ minimum, all $< 0$ maximum, mixed saddle, some $= 0$
inconclusive.

> 🎓 **Exam tip.** Be able to derive "$\Delta\mathbf v = -\eta\nabla c$ decreases $c$" from the linear approximation in
> two lines, and to state why the learning rate cannot be arbitrarily large.

---
## 4 · Two gradient identities you will use constantly

> 📍 **Where we are.** Least squares (Section 5), regression (Module 27), ridge regression and Gaussian likelihoods all
> require gradients of **linear** and **quadratic** functions of a vector. Two identities cover them all. Each is just
> the partial derivatives of Section 2, written compactly with vectors and matrices (Module 08).

**Identity 1 — linear functions.** For $\mathbf x, \mathbf b \in \mathbb R^n$:

$$\nabla_{\mathbf x}\big(\mathbf b^\top\mathbf x\big) = \mathbf b .$$

*In words:* $\mathbf b^\top\mathbf x = b_1x_1 + \cdots + b_nx_n$ is a weighted sum; its slope in direction $x_k$ is just the
weight $b_k$ — the vector version of $\frac{d}{dx}(bx) = b$. **Proof:** $\frac{\partial}{\partial x_k}\sum_i b_ix_i = b_k$,
because only the term $i = k$ contains $x_k$.

**Identity 2 — quadratic forms.** For $\mathbf x \in \mathbb R^n$ and an $n\times n$ matrix $A$:

$$\nabla_{\mathbf x}\big(\mathbf x^\top A\mathbf x\big) = (A + A^\top)\,\mathbf x \;=\; 2A\mathbf x \quad\text{if } A \text{ is symmetric}.$$

*In words:* a quadratic form behaves like $ax^2$ with derivative $2ax$; the matrix version needs both $A$ and $A^\top$
because each $x_k$ appears in row $k$ *and* in column $k$ of the double sum. **Proof:** write
$\mathbf x^\top A\mathbf x = \sum_{i=1}^n\sum_{j=1}^n x_iA_{ij}x_j$. Differentiating with respect to $x_k$ (product rule), the
terms with $i = k$ give $\sum_j A_{kj}x_j = (A\mathbf x)_k$ and the terms with $j = k$ give $\sum_i x_iA_{ik} = (A^\top\mathbf x)_k$:

$$\frac{\partial}{\partial x_k}\sum_{i,j}x_iA_{ij}x_j = \sum_j A_{kj}x_j + \sum_i A_{ik}x_i = \big((A + A^\top)\mathbf x\big)_k .$$

*In words:* the $k$-th partial derivative is the $k$-th entry of $(A + A^\top)\mathbf x$; for symmetric $A$ ($A^\top = A$,
e.g. $A^\top A$ or a covariance matrix) this is $2A\mathbf x$. With $A = I$ we recover $\nabla(\mathbf x^\top\mathbf x) = 2\mathbf x$
of Section 2. For $n = 2$ explicitly: $\mathbf x^\top A\mathbf x = A_{11}x_1^2 + (A_{12} + A_{21})x_1x_2 + A_{22}x_2^2$, whose
$x_1$-derivative is $2A_{11}x_1 + (A_{12} + A_{21})x_2$ — the first entry of $(A + A^\top)\mathbf x$.

⚠️ **Common confusion.** These identities are sometimes printed as "$\nabla_x(bx) = b$" and "$\nabla_x(x^\top xA) = 2Ax$".
The intended expressions are the scalar $\mathbf b^\top\mathbf x$ (a dot product) and the quadratic form
$\mathbf x^\top A\mathbf x$ (a number); $\mathbf x^\top\mathbf x A$ would be a matrix, which has no gradient in this sense.

In [ ]:
# ── 4.1 Check both identities: symbolically for n = 2, numerically for n = 4 with a NON-symmetric A ──
a11, a12, a21, a22, b1, b2 = sp.symbols("a11 a12 a21 a22 b1 b2")
xv = sp.Matrix([X1, X2]); bv = sp.Matrix([b1, b2]); Am = sp.Matrix([[a11, a12], [a21, a22]])
lin = (bv.T * xv)[0]
quad = sp.expand((xv.T * Am * xv)[0])
print("bᵀx =", lin, "  → gradient", [sp.diff(lin, v) for v in (X1, X2)])
print("xᵀAx =", quad)
grad_quad = sp.Matrix([sp.diff(quad, v) for v in (X1, X2)])
print("gradient of xᵀAx − (A + Aᵀ)x =", sp.simplify(grad_quad - (Am + Am.T) * xv).T)

rng4 = np.random.default_rng(SEED + 4)
A4, b4, x4 = rng4.normal(size=(4, 4)), rng4.normal(size=4), rng4.normal(size=4)
print("\nn = 4: ∇(bᵀx) numerical", numerical_gradient(lambda v: b4 @ v, x4).round(6), "| b", b4.round(6))
print("       ∇(xᵀAx) numerical", numerical_gradient(lambda v: v @ A4 @ v, x4).round(6))
print("       (A + Aᵀ)x        ", ((A4 + A4.T) @ x4).round(6))
print("       2Ax (wrong here) ", (2 * A4 @ x4).round(6))
S4 = A4 + A4.T                                                       # a symmetric matrix
print("symmetric S: max |∇(xᵀSx) − 2Sx| =", f"{np.abs(numerical_gradient(lambda v: v @ S4 @ v, x4) - 2 * S4 @ x4).max():.1e}")

🔍 **What just happened?** SymPy differentiates $b_1x_1 + b_2x_2$ to $(b_1, b_2)$ and confirms that the gradient of
$a_{11}x_1^2 + a_{12}x_1x_2 + a_{21}x_1x_2 + a_{22}x_2^2$ minus $(A + A^\top)\mathbf x$ is exactly zero. In four dimensions, with a
random *non-symmetric* $A$, the finite-difference gradient of $\mathbf x^\top A\mathbf x$ matches $(A + A^\top)\mathbf x$ and
visibly differs from $2A\mathbf x$; for the symmetric matrix $S = A + A^\top$ the shortcut $2S\mathbf x$ is correct (difference
$2\times10^{-10}$, pure rounding).

---
## 5 · Application: least-squares minimisation

> 📍 **Where we are.** Module 09 solved $A\mathbf x = \mathbf b$ when a solution exists; Module 11 explained when it does:
> exactly when $\mathbf b$ lies in the **span of the columns** of $A$. With more measurements than unknowns that almost
> never happens — biological measurements are noisy. Calculus finds the next best thing.

**The problem.** We have $A\mathbf x = \mathbf b$ with $A \in \mathbb R^{m\times n}$ and $\mathbf b \in \mathbb R^{m}$ ($m$
equations, $n$ unknowns). If $\mathbf b \notin \operatorname{span}(\text{columns}(A))$, then there is **no**
$\mathbf x \in \mathbb R^n$ with $A\mathbf x = \mathbf b$. But we can find something "close enough": the $\mathbf x$ that makes the
**squared error** smallest,

$$\min_{\mathbf x}\;\lVert A\mathbf x - \mathbf b\rVert_2^2 .$$

*In words:* $A\mathbf x - \mathbf b$ is the vector of misfits (residuals) of all $m$ equations; we minimise the sum of their
squares. (⚠️ The right-hand side $\mathbf b$ has one entry per *equation*, so it lives in $\mathbb R^m$, not
$\mathbb R^n$, and the unknown $\mathbf x$ lives in $\mathbb R^n$.)

**Norm squared.** For any vector $\mathbf x = (x_1, \dots, x_n)^\top \in \mathbb R^n$,

$$\lVert\mathbf x\rVert_2^2 = \sum_{i=1}^n x_i^2 = \mathbf x^\top\mathbf x .$$

*In words:* the squared length is the sum of squared entries (Module 07), and that sum is exactly the matrix product of
the row $\mathbf x^\top = [x_1 \cdots x_n]$ with the column $\mathbf x$: $\mathbf x^\top\mathbf x = x_1x_1 + x_2x_2 + \cdots + x_nx_n$.

**Expand the squared error** (using $(PQ)^\top = Q^\top P^\top$, Module 08):

$$
\begin{aligned}
\lVert A\mathbf x - \mathbf b\rVert_2^2 &= (A\mathbf x - \mathbf b)^\top(A\mathbf x - \mathbf b) && \text{norm squared}\\
&= (\mathbf x^\top A^\top - \mathbf b^\top)(A\mathbf x - \mathbf b) && \text{transpose of a difference; } (A\mathbf x)^\top = \mathbf x^\top A^\top\\
&= \mathbf x^\top A^\top A\mathbf x - \mathbf x^\top A^\top\mathbf b - \mathbf b^\top A\mathbf x + \mathbf b^\top\mathbf b && \text{multiply out}\\
&= \mathbf x^\top A^\top A\mathbf x - 2\,\mathbf x^\top A^\top\mathbf b + \mathbf b^\top\mathbf b && \mathbf b^\top A\mathbf x \text{ is a number, equal to its transpose } \mathbf x^\top A^\top\mathbf b
\end{aligned}
$$

*In words:* a quadratic term, a linear term and a constant — the vector version of $(ax - b)^2 = a^2x^2 - 2abx + b^2$.
(⚠️ A middle term written as "$2A\mathbf x\mathbf b^\top$" cannot be right: $A\mathbf x\,\mathbf b^\top$ is an $m\times m$ matrix,
not a number.)

**The gradient** with respect to $\mathbf x$, by the identities of Section 4 (identity 2 with the symmetric matrix
$A^\top A$, identity 1 with the vector $A^\top\mathbf b$; the constant $\mathbf b^\top\mathbf b$ has gradient $\mathbf 0$):

$$\nabla_{\mathbf x}\lVert A\mathbf x - \mathbf b\rVert_2^2 = 2A^\top A\mathbf x - 2A^\top\mathbf b .$$

**Set it to zero:** $2A^\top A\mathbf x - 2A^\top\mathbf b = \mathbf 0 \;\Longrightarrow\; A^\top A\mathbf x = A^\top\mathbf b$ — the
**normal equations**. Solving for $\mathbf x$ (assuming $A^\top A$ is invertible, which holds exactly when the columns of $A$
are linearly independent, Module 11):

$$\mathbf x = (A^\top A)^{-1}A^\top\mathbf b .$$

*In words:* the least-squares solution solves a small $n\times n$ system; it is the global minimum because the Hessian
$2A^\top A$ has no negative eigenvalues (Section 3.5; $\mathbf z^\top A^\top A\mathbf z = \lVert A\mathbf z\rVert^2 \ge 0$, Module 12).
**Geometry:** the normal equations say $A^\top(\mathbf b - A\mathbf x) = \mathbf 0$ — the residual is **orthogonal to every column
of $A$**, so $A\mathbf x$ is the orthogonal projection of $\mathbf b$ onto the column space: the closest point to $\mathbf b$ that
the model can reach.

In [ ]:
# ── 5.1 A toy over-determined system: a straight line through three measurements ──
# Expression of a reporter gene at t = 0, 1, 2 h is 1, 3, 2 (a.u.); model y = x1 + x2·t → three equations, two unknowns
A_toy = np.array([[1.0, 0.0], [1.0, 1.0], [1.0, 2.0]])
b_toy = np.array([1.0, 3.0, 2.0])
print("rank A =", np.linalg.matrix_rank(A_toy), "| rank [A | b] =", np.linalg.matrix_rank(np.column_stack([A_toy, b_toy])),
      "→ b is not in the column space: no exact solution")
A_s, b_s = sp.Matrix(A_toy.astype(int)), sp.Matrix(b_toy.astype(int))
AtA, Atb = A_s.T * A_s, A_s.T * b_s
x_exact = AtA.inv() * Atb
print("AᵀA =", AtA.tolist(), " Aᵀb =", Atb.T.tolist(), " det(AᵀA) =", AtA.det())
print("x = (AᵀA)⁻¹Aᵀb =", x_exact.T.tolist(), "=", [float(v) for v in x_exact])
x_ls = np.linalg.lstsq(A_toy, b_toy, rcond=None)[0]
resid = b_toy - A_toy @ x_ls
sq_err = lambda v: np.sum((A_toy @ v - b_toy) ** 2)
print("np.linalg.lstsq:", x_ls.round(6), "| residual b − Ax =", resid.round(4), "| Aᵀ·residual =", (A_toy.T @ resid).round(12) + 0.0)
print("gradient 2AᵀAx − 2Aᵀb at the solution:", (2 * A_toy.T @ A_toy @ x_ls - 2 * A_toy.T @ b_toy).round(12) + 0.0,
      "| numerical:", numerical_gradient(sq_err, x_ls).round(6) + 0.0)

fig = plt.figure(figsize=(11, 4))
ax1 = fig.add_subplot(1, 2, 1)
tt = np.linspace(-0.2, 2.2, 50)
ax1.plot([0, 1, 2], b_toy, "ko", ms=7, label="measurements b")
ax1.plot(tt, x_ls[0] + x_ls[1] * tt, color="tab:red", label=f"least squares: y = {x_ls[0]:.3f} + {x_ls[1]:.2f} t")
for ti, bi, pi in zip([0, 1, 2], b_toy, A_toy @ x_ls):
    ax1.plot([ti, ti], [bi, pi], color="tab:blue", lw=2)
ax1.set_xlabel("time t (h)"); ax1.set_ylabel("expression (a.u.)"); ax1.legend(); ax1.set_title("residuals (blue) are vertical misfits")
ax2 = fig.add_subplot(1, 2, 2, projection="3d", computed_zorder=False)
s_, t_ = np.meshgrid(np.linspace(-0.3, 2.6, 2), np.linspace(-0.6, 1.2, 2))
plane = s_[..., None] * A_toy[:, 0] + t_[..., None] * A_toy[:, 1]       # points s·(column 1) + t·(column 2)
ax2.plot_surface(plane[..., 0], plane[..., 1], plane[..., 2], alpha=0.3, color="tab:green", zorder=1)
proj = A_toy @ x_ls
ax2.plot([0, b_toy[0]], [0, b_toy[1]], [0, b_toy[2]], color="k", lw=2.5, zorder=5, label="b (the measurements)")
ax2.plot([0, proj[0]], [0, proj[1]], [0, proj[2]], color="tab:red", lw=2.5, zorder=5, label="Ax* (closest point in the plane)")
ax2.plot([proj[0], b_toy[0]], [proj[1], b_toy[1]], [proj[2], b_toy[2]], color="tab:blue", lw=2.5, zorder=6,
         label="residual b − Ax* (perpendicular)")
ax2.scatter(*b_toy, color="k", s=25, zorder=7); ax2.scatter(*proj, color="tab:red", s=25, zorder=7)
ax2.set_xlabel("eq. 1"); ax2.set_ylabel("eq. 2"); ax2.set_zlabel("eq. 3"); ax2.view_init(elev=8, azim=-118)
ax2.legend(fontsize=7, loc="upper left"); ax2.set_title("column space of A (green plane) in ℝ³", fontsize=9)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The augmented matrix $[A\mid\mathbf b]$ has rank 3 but $A$ only rank 2, so $\mathbf b$ is outside the
column space and the three equations have no common solution (Modules 09 and 11). The normal equations are
$A^\top A = \begin{bmatrix}3 & 3\\ 3 & 5\end{bmatrix}$, $A^\top\mathbf b = (6, 7)$, $\det(A^\top A) = 6$, and give exactly
$\mathbf x = (3/2, 1/2)$: the line $y = 1.5 + 0.5t$. NumPy's `lstsq` agrees; the residual $(-0.5, 1, -0.5)$ is orthogonal to
both columns ($A^\top\mathbf r = \mathbf 0$: $-0.5 + 1 - 0.5 = 0$ and $0 + 1 - 1 = 0$), and the gradient vanishes there both by
formula and numerically.
In the left panel the residuals are the vertical misfits whose squares we minimised. In the right panel the same
fit is seen in $\mathbb R^3$ (one axis per equation): the model can only reach points of the green plane (the column
space), and the best reachable point $A\mathbf x^\ast$ is the foot of the perpendicular from $\mathbf b$.

Now a real-sized problem: the **insulin study**. Each of the 120 adults gives one equation
$\beta_0 + \beta_1\,\text{BMI}_i + \beta_2\,\text{age}_i + \beta_3\,\text{glucose}_i \approx \text{insulin}_i$, so $A$ is
$120\times4$ (a column of ones for the intercept) and $\mathbf x = (\beta_0, \beta_1, \beta_2, \beta_3)$.

In [ ]:
# ── 5.2 The SIMULATED insulin study: normal equations, lstsq, the truth, and gradient descent ──
A_ins = np.column_stack([np.ones(len(insulin)), insulin[["bmi", "age", "glucose"]].to_numpy(float)])
b_ins = insulin["insulin"].to_numpy(float)
beta_ne = np.linalg.solve(A_ins.T @ A_ins, A_ins.T @ b_ins)           # normal equations (solve, never invert)
beta_ls = np.linalg.lstsq(A_ins, b_ins, rcond=None)[0]
names = ["intercept", "bmi", "age", "glucose"]
print(pd.DataFrame({"normal equations": beta_ne, "np.linalg.lstsq": beta_ls,
                    "truth (simulator)": [INSULIN_TRUE_COEFFICIENTS[k] for k in names]}, index=names).round(4).to_string())
r_ins = b_ins - A_ins @ beta_ne
print("Aᵀ·residual (should be 0):", (A_ins.T @ r_ins).round(8) + 0.0,
      f"| residual SD {r_ins.std(ddof=4):.3f} (true σ = {INSULIN_TRUE_SIGMA})")

# gradient descent on the mean squared error (1/m)‖Ax − b‖², raw predictors vs standardised predictors
def mse_gd(A, b, n_steps):
    """Gradient descent on (1/m)‖Ax − b‖² with the safe step 1/λmax of its Hessian (2/m)AᵀA; returns the cost history."""
    m = len(b)
    eta_ = 1.0 / np.linalg.eigvalsh(2 * A.T @ A / m).max()
    x_ = np.zeros(A.shape[1]); costs = []
    for _ in range(n_steps):
        costs.append(np.mean((A @ x_ - b) ** 2))
        x_ = x_ - eta_ * 2 * A.T @ (A @ x_ - b) / m
    return x_, np.array(costs)


means, sds = A_ins[:, 1:].mean(axis=0), A_ins[:, 1:].std(axis=0)
A_std = np.column_stack([np.ones(len(b_ins)), (A_ins[:, 1:] - means) / sds])
best_mse = np.mean(r_ins ** 2)
x_raw, cost_raw = mse_gd(A_ins, b_ins, 5000)
x_std, cost_std = mse_gd(A_std, b_ins, 5000)
beta_from_std = np.r_[x_std[0] - np.sum(x_std[1:] * means / sds), x_std[1:] / sds]   # back to original units
print(f"condition number of AᵀA: raw {np.linalg.cond(A_ins.T @ A_ins):.2e}, standardised {np.linalg.cond(A_std.T @ A_std):.2f}")
print("after 5000 steps, raw predictors       :", x_raw.round(4), f"(MSE gap {cost_raw[-1] - best_mse:.2e})")
print("after 5000 steps, standardised (→ units):", beta_from_std.round(4), f"(MSE gap {cost_std[-1] - best_mse:.1e})")
print("steps until the MSE gap < 1e-10 (standardised):", int(np.argmax(cost_std - best_mse < 1e-10)))

fig, ax = plt.subplots(figsize=(6.4, 3.4))
steps = np.arange(1, len(cost_raw) + 1)
ax.loglog(steps, np.maximum(cost_raw - best_mse, 1e-16), label="raw predictors (BMI, age, glucose)")
ax.loglog(steps, np.maximum(cost_std - best_mse, 1e-16), label="standardised predictors")
ax.set_xlabel("gradient-descent step"); ax.set_ylabel("MSE − minimum MSE")
ax.set_title("SIMULATED insulin study: same problem, very different speed"); ax.legend(); plt.tight_layout(); plt.show()

🔍 **What just happened?** The normal equations and `lstsq` give the same coefficients — intercept $-2.94$, BMI 0.563, age
0.034, glucose 0.051 per unit — close to the simulator's truth ($-4$, 0.55, 0.04, 0.06); the differences are the
sampling noise of 120 people (Module 27 attaches standard errors to them). The residual is orthogonal to all four
columns (numerically zero) and its SD (2.69) is close to the true noise SD 2.5.

Gradient descent tells a cautionary story. With the raw predictors (BMI ≈ 27, age ≈ 50, glucose ≈ 90) the matrix
$A^\top A$ has condition number about $10^6$: its largest Hessian eigenvalue forces a tiny step, while the smallest
eigenvalue's direction (essentially the intercept) barely moves — after 5,000 steps the intercept is still near 0
instead of $-2.94$. After **standardising** each predictor (subtract the mean, divide by the SD) the condition number drops
to 3.1, and the same algorithm gets within $10^{-10}$ of the minimum in 30 steps; converting the coefficients
back to original units reproduces the normal-equation solution. This is why every machine-learning pipeline
standardises its features (BiologyToML 04, 06, 22).

✏️ **Your turn.** (1) Add the column `bmi * glucose` (an interaction) to `A_ins` and refit with the normal equations.
Does the coefficient of the interaction look different from 0? (2) Make the design rank-deficient by adding a column
`2 * bmi`. What does `np.linalg.cond(A.T @ A)` say, and what happens to `np.linalg.solve`? (Module 11: dependent columns
⇒ $A^\top A$ is singular.)

### 📘 Lecture notes — gradient identities and least squares

**Identities** ($\mathbf x, \mathbf b \in \mathbb R^n$): $\nabla_{\mathbf x}(\mathbf b^\top\mathbf x) = \mathbf b$;
$\nabla_{\mathbf x}(\mathbf x^\top A\mathbf x) = (A + A^\top)\mathbf x$, $= 2A\mathbf x$ for symmetric $A$; special case
$\nabla_{\mathbf x}(\mathbf x^\top\mathbf x) = 2\mathbf x$. Proof by components: $\partial_k\sum_{i,j}x_iA_{ij}x_j = (A\mathbf x)_k + (A^\top\mathbf x)_k$.

**Least squares.** $A\mathbf x = \mathbf b$, $A \in \mathbb R^{m\times n}$, $\mathbf b \in \mathbb R^m$, $\mathbf b \notin \operatorname{span}(\operatorname{col}(A))$
⇒ $\nexists\,\mathbf x \in \mathbb R^n$ with $A\mathbf x = \mathbf b$. Minimise $\lVert A\mathbf x - \mathbf b\rVert_2^2$ (squared error).
With $\lVert\mathbf x\rVert_2^2 = \sum_i x_i^2 = \mathbf x^\top\mathbf x$:
$\lVert A\mathbf x - \mathbf b\rVert_2^2 = (A\mathbf x - \mathbf b)^\top(A\mathbf x - \mathbf b) = (\mathbf x^\top A^\top - \mathbf b^\top)(A\mathbf x - \mathbf b) = \mathbf x^\top A^\top A\mathbf x - 2\mathbf x^\top A^\top\mathbf b + \mathbf b^\top\mathbf b$.
Gradient $2A^\top A\mathbf x - 2A^\top\mathbf b = \mathbf 0$ ⇒ **normal equations** $A^\top A\mathbf x = A^\top\mathbf b$ ⇒
$\mathbf x = (A^\top A)^{-1}A^\top\mathbf b$ (if $A^\top A$ is invertible ⇔ columns of $A$ independent).

**Geometry.** Residual $\mathbf b - A\mathbf x^\ast$ ⟂ every column of $A$; $A\mathbf x^\ast$ = projection of $\mathbf b$ onto the column
space. Convex: Hessian $2A^\top A$ is positive semidefinite.

**Worked example.** Points $(0,1), (1,3), (2,2)$: $A^\top A = \begin{bmatrix}3&3\\3&5\end{bmatrix}$, $A^\top\mathbf b = (6,7)$, $\mathbf x = (3/2, 1/2)$, residual $(-\tfrac12, 1, -\tfrac12)$.

> 🎓 **Exam tip.** Reproduce the four-line expansion, including *why* the two middle terms combine (a scalar equals
> its transpose), and the order of factors in $(A^\top A)^{-1}A^\top\mathbf b$. In code, solve the normal equations with
> `np.linalg.solve` or use `lstsq` — never form the inverse explicitly.

### 🌉 Bridge — least squares from genome-wide association to cryo-EM

The normal equations are the workhorse of quantitative biology. **Linear regression** (Module 27; BiologyToML
[06 · linear regression and GWAS](../../../../ml/tutorial/BiologyToML/notebooks/06_linear_regression_gwas.ipynb)) fits
one SNP at a time against a phenotype in a genome-wide association study — millions of tiny least-squares problems.
**Ridge regression** (BiologyToML [07](../../../../ml/tutorial/BiologyToML/notebooks/07_ridge_lasso_sparse_regression.ipynb))
adds $\lambda\mathbf x^\top\mathbf x$ to the squared error; identity 2 turns the normal equations into
$(A^\top A + \lambda I)\mathbf x = A^\top\mathbf b$, solvable even when there are more genes than samples. **Cell-type
deconvolution** of bulk RNA-seq solves (non-negative) least squares between a signature matrix and a bulk profile
(Module 09). And when the model is not linear in its parameters — enzyme kinetics, dose–response, growth curves —
the Jacobian of Section 6 turns the fit into a sequence of least-squares problems (Section 7).

---
## 6 · The Jacobian: many inputs, many outputs

> 📍 **Where we are.** The gradient handles many inputs and **one** output. Sometimes we also have functions with
> several outputs: the residuals of all measurements of a fit (Section 7), the rates of change of all species of a
> dynamical system (Sections 8–10), the activations of a whole layer of a neural network. Their derivative is a
> **matrix**.

**Definition.** For a function $g:\mathbb R^n \to \mathbb R^m$ with components $g_1, \dots, g_m$ (each a function of
$\mathbf x = (x_1, \dots, x_n)$), the **Jacobian** of $g$ is the $m\times n$ matrix of all partial derivatives

$$J_g(\mathbf x) = \begin{bmatrix}
\dfrac{\partial g_1}{\partial x_1} & \dfrac{\partial g_1}{\partial x_2} & \cdots & \dfrac{\partial g_1}{\partial x_n}\\[2mm]
\dfrac{\partial g_2}{\partial x_1} & \dfrac{\partial g_2}{\partial x_2} & \cdots & \dfrac{\partial g_2}{\partial x_n}\\
\vdots & \vdots & \ddots & \vdots\\
\dfrac{\partial g_m}{\partial x_1} & \dfrac{\partial g_m}{\partial x_2} & \cdots & \dfrac{\partial g_m}{\partial x_n}
\end{bmatrix} .$$

*In words:* row $i$ is the gradient of the $i$-th output (written as a row), column $j$ says how all outputs respond to
the $j$-th input; entry $(i, j)$ is "how much output $i$ changes per unit change of input $j$". Special cases: $m = 1$
gives the gradient as a $1\times n$ row; $m = n = 1$ gives the ordinary derivative. ($m$ and $n$ are positive
**integers** — numbers of outputs and inputs — and the definition works for any $m, n \ge 1$.)

**Worked example.** $h:\mathbb R^2 \to \mathbb R^2$,

$$h(\mathbf x) = \mathbf x^\top\begin{bmatrix}3 & 1\\ 0 & -1\end{bmatrix}\begin{bmatrix}x_1 & 0\\ 0 & x_2\end{bmatrix}
= \mathbf x^\top\begin{bmatrix}3x_1 & x_2\\ 0 & -x_2\end{bmatrix} = \begin{bmatrix}3x_1^2 & x_1x_2 - x_2^2\end{bmatrix},
\qquad\text{i.e.}\qquad h(\mathbf x) = \begin{pmatrix}3x_1^2\\ x_1x_2 - x_2^2\end{pmatrix}.$$

*In words:* multiply the two matrices first ($3\cdot x_1 + 1\cdot 0$, $3\cdot0 + 1\cdot x_2$, …), then multiply the row
$\mathbf x^\top = [x_1\; x_2]$ into the result: $x_1\cdot 3x_1 + x_2\cdot 0 = 3x_1^2$ and $x_1x_2 + x_2(-x_2) = x_1x_2 - x_2^2$.
(⚠️ A row vector times a matrix gives a **row** vector; we then write $h$ as a column, as usual for vector-valued functions.
Mixing the two shapes in one equation is a common slip.) The Jacobian:

$$J_h(\mathbf x) = \begin{bmatrix}\dfrac{\partial(3x_1^2)}{\partial x_1} & \dfrac{\partial(3x_1^2)}{\partial x_2}\\[2mm]
\dfrac{\partial(x_1x_2 - x_2^2)}{\partial x_1} & \dfrac{\partial(x_1x_2 - x_2^2)}{\partial x_2}\end{bmatrix}
= \begin{bmatrix}6x_1 & 0\\ x_2 & x_1 - 2x_2\end{bmatrix} .$$

*In words:* the first output depends only on $x_1$ (slope $6x_1$, and 0 for $x_2$); the second has slope $x_2$ in the
$x_1$ direction and $x_1 - 2x_2$ in the $x_2$ direction (product rule and power rule). Every entry is a *partial*
derivative, $\partial/\partial x_j$.

In [ ]:
# ── 6.1 The worked example in SymPy, and a numerical Jacobian ──
B_ex = sp.Matrix([[3, 1], [0, -1]])
middle = B_ex * sp.diag(X1, X2)
row_form = xv.T * middle
print("B·diag(x1, x2) =", middle.tolist())
print("xᵀ·B·diag(x) =", row_form.tolist(), "  shape", row_form.shape, "(a ROW vector)")
h_col = row_form.T                                                    # write h as a column vector
J_h = h_col.jacobian(xv)
print("J_h(x) =", J_h.tolist())


def numerical_jacobian(g, x, h=1e-6):
    """m × n Jacobian of g: ℝⁿ → ℝᵐ by central differences: column j = (g(x + h·e_j) − g(x − h·e_j)) / 2h."""
    x = np.asarray(x, dtype=float)
    cols = []
    for j in range(x.size):
        step = np.zeros_like(x)
        step[j] = h
        cols.append((np.asarray(g(x + step)) - np.asarray(g(x - step))) / (2 * h))
    return np.column_stack(cols)


h_fun = lambda v: np.array([3 * v[0] ** 2, v[0] * v[1] - v[1] ** 2])
x0_ = np.array([1.0, 2.0])
print("at x = (1, 2): formula", np.array(J_h.subs({X1: 1, X2: 2})).astype(float).tolist(),
      "| numerical", numerical_jacobian(h_fun, x0_).round(6).tolist())

🔍 **What just happened?** SymPy reproduces the worked example step by step: $B\,\mathrm{diag}(x_1, x_2) = [[3x_1, x_2], [0, -x_2]]$,
the product with $\mathbf x^\top$ is the $1\times 2$ **row** $[3x_1^2,\; x_1x_2 - x_2^2]$, and the Jacobian of $h$ (as a column)
is $[[6x_1, 0], [x_2, x_1 - 2x_2]]$. At $\mathbf x = (1, 2)$ it is $[[6, 0], [2, -3]]$, and the finite-difference Jacobian agrees.

**Why we need the Jacobian.** For a vector map $g:\mathbb R^n\to\mathbb R^m$, $J_g(\mathbf x)$ captures how small input changes
move the outputs.

* **Local linearisation.** Near $\mathbf x$:
  $$g(\mathbf x + d\mathbf x) \approx g(\mathbf x) + J_g(\mathbf x)\,d\mathbf x,$$
  *in words:* $J_g(\mathbf x)$ maps small input changes $d\mathbf x$ to output changes $d\mathbf y$ — the multi-output
  generalisation of the gradient and of $f(x + \varepsilon) \approx f(x) + \varepsilon f'(x)$. Up close, every smooth map looks
  like a matrix (Module 08: a linear map), and $\lvert\det J\rvert$ is the factor by which it scales small areas.
* **Chain rule for multivariable compositions.** If $\mathbf y = g(\mathbf x)$ and $\mathbf z = h(\mathbf y)$, then
  $$d\mathbf z = J_h(\mathbf y)\,J_g(\mathbf x)\,d\mathbf x, \qquad\text{i.e.}\qquad J_{h\circ g}(\mathbf x) = J_h\big(g(\mathbf x)\big)\,J_g(\mathbf x),$$
  *in words:* the Jacobian of a composition is the **matrix product** of the Jacobians (inside first on the right) —
  Module 14's $f'(g(x))\,g'(x)$ with matrices. This is the matrix form used for gradient flow through the layers of a
  neural network.
* **Nonlinear least squares** (Section 7): with residuals $\mathbf r(\mathbf x)$ and $J = \partial\mathbf r/\partial\mathbf x$, the
  update solves $(J^\top J)\,\Delta\mathbf x = -J^\top\mathbf r$: the Jacobian provides the directions and weights for solving
  multi-output fitting problems.

In [ ]:
# ── 6.2 Local linearisation of h around x0 = (1, 2): small squares become (almost) parallelograms ──
J0 = numerical_jacobian(h_fun, x0_)
for s in [0.5, 0.1, 0.01, 0.001]:
    dx = s * np.array([1.0, -0.6])
    err = np.linalg.norm(h_fun(x0_ + dx) - (h_fun(x0_) + J0 @ dx))
    print(f"‖dx‖ = {np.linalg.norm(dx):.4f}: ‖h(x0 + dx) − [h(x0) + J dx]‖ = {err:.2e}   (error / ‖dx‖² = {err / np.linalg.norm(dx) ** 2:.3f})")
print(f"det J(x0) = {np.linalg.det(J0):.1f}: small areas are scaled by |det J| = {abs(np.linalg.det(J0)):.0f} and flipped (det < 0)")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
offs = np.linspace(-0.4, 0.4, 9)
for k, o in enumerate(offs):
    for line in (np.c_[x0_[0] + offs, np.full(9, x0_[1] + o)], np.c_[np.full(9, x0_[0] + o), x0_[1] + offs]):
        fine = np.c_[np.interp(np.linspace(0, 8, 60), range(9), line[:, 0]), np.interp(np.linspace(0, 8, 60), range(9), line[:, 1])]
        axes[0].plot(fine[:, 0], fine[:, 1], color="tab:blue", lw=0.8)
        img = np.array([h_fun(p) for p in fine])
        lin = h_fun(x0_) + (fine - x0_) @ J0.T
        axes[1].plot(img[:, 0], img[:, 1], color="tab:blue", lw=0.8)
        axes[1].plot(lin[:, 0], lin[:, 1], color="tab:red", lw=0.8, ls="--")
axes[0].plot(*x0_, "k*", ms=12); axes[1].plot(*h_fun(x0_), "k*", ms=12)
axes[0].set_title("input grid around x0 = (1, 2)"); axes[0].set_xlabel("x1"); axes[0].set_ylabel("x2"); axes[0].set_aspect("equal")
axes[1].set_title("image under h (blue) vs h(x0) + J·dx (red dashed)", fontsize=9)
axes[1].set_xlabel("h1 = 3x1²"); axes[1].set_ylabel("h2 = x1x2 − x2²")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The error of the linear approximation shrinks like the *square* of the step (the ratio
error/‖dx‖² stays at 2.316): the Jacobian is the best linear description of $h$ near $\mathbf x_0$. In the pictures, the
square input grid is bent by $h$ into curved cells (blue), but near the centre the cells are almost exactly the
parallelograms produced by the matrix $J$ (red dashed); far from the centre the curvature shows. Since
$\det J(\mathbf x_0) = -18$, tiny squares near $(1, 2)$ come out 18 times larger in area and mirror-flipped — Module 08's
geometric meaning of the determinant, now *local*.

In [ ]:
# ── 6.3 The multivariable chain rule J_(h∘g) = J_h(g(x)) · J_g(x) — and back-propagation through one neuron ──
g_fun = lambda v: np.array([v[0] * v[1], v[0] + v[1] ** 2])           # an inner map g: ℝ² → ℝ²
J_g = lambda v: np.array([[v[1], v[0]], [1.0, 2 * v[1]]])
J_h_fun = lambda y_: np.array([[6 * y_[0], 0.0], [y_[1], y_[0] - 2 * y_[1]]])
xc = np.array([0.5, -1.0])
print("numerical Jacobian of h∘g  :", numerical_jacobian(lambda v: h_fun(g_fun(v)), xc).round(6).tolist())
print("product J_h(g(x)) · J_g(x) :", (J_h_fun(g_fun(xc)) @ J_g(xc)).round(6).tolist())

# the neuron of Section 3.4 is a composition (w, b) → z = w·x + b → a = σ(z) → c = (1/2m)‖f − a‖²
v_chk = np.array([-1.0, 0.5])
z_ = v_chk[0] * log_dose + v_chk[1]; a_ = logistic(z_); m_ = len(log_dose)
J_z = np.column_stack([log_dose, np.ones(m_)])                        # m × 2
J_a = np.diag(a_ * (1 - a_))                                          # m × m (each a_j depends only on z_j)
J_c = ((a_ - viability) / m_)[None, :]                                # 1 × m
print("product of three Jacobians J_c · J_a · J_z :", (J_c @ J_a @ J_z).round(8).ravel(),
      "\nneuron_grad from Section 3.4             :", neuron_grad(v_chk).round(8))

🔍 **What just happened?** For the composition $h\circ g$ the numerical Jacobian equals the matrix product
$J_h(g(\mathbf x))\,J_g(\mathbf x)$, entry by entry. The second check rewrites the one-neuron gradient of Section 3.4 as a
product of three Jacobians — cost w.r.t. outputs ($1\times m$), outputs w.r.t. pre-activations ($m\times m$, diagonal) and
pre-activations w.r.t. parameters ($m\times 2$) — and gets the same two numbers. A deep network is a long composition of
layers, so its gradient is a long product of Jacobians; **back-propagation** evaluates that product from the cost
backwards, because multiplying a row vector into one matrix after another is much cheaper than multiplying the big
matrices together.

✏️ **Your turn.** (1) Compute by hand the Jacobian of $g(\mathbf x) = (x_1e^{x_2},\; x_1^2 + x_2)$ and check it with
`numerical_jacobian` at $(1, 0)$. (2) For a *linear* map $g(\mathbf x) = M\mathbf x + \mathbf c$, what is $J_g$? (This
answer is the key to Section 8.)

### 📘 Lecture notes — the Jacobian

**Definition.** For $g:\mathbb R^n\to\mathbb R^m$ ($m, n$ positive integers), $J_g(\mathbf x)$ is the $m\times n$ matrix with
entries $\partial g_i/\partial x_j$ (rows = outputs, columns = inputs). $m = 1$: the gradient (as a row).

**Worked example.** $h(\mathbf x) = \mathbf x^\top\begin{bmatrix}3&1\\0&-1\end{bmatrix}\begin{bmatrix}x_1&0\\0&x_2\end{bmatrix} = \mathbf x^\top\begin{bmatrix}3x_1&x_2\\0&-x_2\end{bmatrix}$,
i.e. $h(\mathbf x) = (3x_1^2,\; x_1x_2 - x_2^2)^\top$; $J_h(\mathbf x) = \begin{bmatrix}6x_1 & 0\\ x_2 & x_1 - 2x_2\end{bmatrix}$
($[[6, 0], [2, -3]]$ at $(1, 2)$).

**Why we need it.** Local linearisation $g(\mathbf x + d\mathbf x) \approx g(\mathbf x) + J_g(\mathbf x)d\mathbf x$ (small input
changes → output changes; the multi-output gradient). Chain rule $d\mathbf z = J_h(\mathbf y)J_g(\mathbf x)d\mathbf x$ for
$\mathbf y = g(\mathbf x)$, $\mathbf z = h(\mathbf y)$ — the matrix form of gradient flow through layers (back-propagation).
Nonlinear least squares $(J^\top J)\Delta\mathbf x = -J^\top\mathbf r$. The Jacobian of a linear map $M\mathbf x + \mathbf c$ is $M$.
The Jacobian and the **Hessian** (Section 3.5: the Jacobian of the gradient) drive advanced optimisation and modelling.

> 🎓 **Exam tip.** Keep the shapes straight: $J_g$ has one **row per output** and one **column per input**; in a chain
> rule the inner function's Jacobian stands on the **right**. A quick shape check catches most errors.

---
## 7 · Nonlinear least squares: fitting enzyme kinetics with the Jacobian

> 📍 **Where we are.** Section 5 solved least squares when the model is **linear** in its parameters. The
> Michaelis–Menten model $v = \frac{V_{max}S}{K_m + S}$ is not linear in $K_m$, so there are no normal equations to solve
> once. The Jacobian lets us *linearise around the current guess*, solve a linear least-squares problem for the
> correction, and repeat.

Let $\boldsymbol\theta = (V_{max}, K_m)$ and define the residuals $r_j(\boldsymbol\theta) = \frac{V_{max}S_j}{K_m + S_j} - v_j$ for the
$m = 16$ measurements. We minimise $\frac12\lVert\mathbf r(\boldsymbol\theta)\rVert^2$. Near the current guess, the Jacobian
$J = \partial\mathbf r/\partial\boldsymbol\theta$ ($16\times 2$) gives $\mathbf r(\boldsymbol\theta + \Delta\boldsymbol\theta) \approx \mathbf r + J\Delta\boldsymbol\theta$,
and minimising $\lVert\mathbf r + J\Delta\boldsymbol\theta\rVert^2$ over $\Delta\boldsymbol\theta$ is Section 5's problem with $A = J$ and
$\mathbf b = -\mathbf r$. Its normal equations are the **Gauss–Newton** update

$$(J^\top J)\,\Delta\boldsymbol\theta = -J^\top\mathbf r, \qquad \boldsymbol\theta \leftarrow \boldsymbol\theta + \Delta\boldsymbol\theta,$$

*in words:* linearise, solve the small linear least-squares problem, step, repeat. The Jacobian columns are
$\frac{\partial r_j}{\partial V_{max}} = \frac{S_j}{K_m + S_j}$ and $\frac{\partial r_j}{\partial K_m} = -\frac{V_{max}S_j}{(K_m + S_j)^2}$
(quotient rule). The method is called Gauss–*Newton* because $J^\top J$ approximates the Hessian of
$\frac12\lVert\mathbf r\rVert^2$ (Newton's method would use the full Hessian, which has an extra term with the second
derivatives of $\mathbf r$); near a good fit with small residuals the two are almost equal and convergence is fast.

A classic way to get a starting guess is the **Lineweaver–Burk** (double-reciprocal) plot: since
$\frac1v = \frac{K_m}{V_{max}}\cdot\frac1S + \frac{1}{V_{max}}$, a straight line through $(1/S, 1/v)$ gives intercept $1/V_{max}$ and
slope $K_m/V_{max}$.

In [ ]:
# ── 7.1 Gauss–Newton for Michaelis–Menten on the SIMULATED enzyme data, started from Lineweaver–Burk ──
def mm_residual(theta, S=S_conc, v=v_obs):
    """Residuals r_j = Vmax·S_j/(Km + S_j) − v_j."""
    return theta[0] * S / (theta[1] + S) - v


def mm_jacobian(theta, S=S_conc):
    """16 × 2 Jacobian of the residuals: columns ∂r/∂Vmax = S/(Km + S), ∂r/∂Km = −Vmax·S/(Km + S)²."""
    return np.column_stack([S / (theta[1] + S), -theta[0] * S / (theta[1] + S) ** 2])


lb_slope, lb_icpt = np.polyfit(1 / S_conc, 1 / v_obs, 1)
theta = np.array([1 / lb_icpt, lb_slope / lb_icpt])
print(f"Lineweaver–Burk estimate: Vmax = {theta[0]:.3f}, Km = {theta[1]:.3f}   (truth {MM_TRUE['Vmax']}, {MM_TRUE['Km']})")
print("max |Jacobian − finite differences| =", f"{np.abs(mm_jacobian(theta) - numerical_jacobian(mm_residual, theta)).max():.1e}")
for it in range(6):
    r_, J_ = mm_residual(theta), mm_jacobian(theta)
    step = np.linalg.solve(J_.T @ J_, -J_.T @ r_)
    theta = theta + step
    print(f"iteration {it + 1}: Vmax = {theta[0]:.5f}, Km = {theta[1]:.5f}, ½‖r‖² = {0.5 * np.sum(mm_residual(theta) ** 2):.6f}")
popt, _ = optimize.curve_fit(lambda S, Vm, K: Vm * S / (K + S), S_conc, v_obs, p0=[10, 2])
print(f"scipy.optimize.curve_fit: Vmax = {popt[0]:.5f}, Km = {popt[1]:.5f}")

🔍 **What just happened?** The Lineweaver–Burk line gives $V_{max} \approx 8.2$ and $K_m \approx 1.6$ — far below the truth
(12 and 3). Starting from there, Gauss–Newton moves to $V_{max} = 11.85$, $K_m = 2.77$ and stops changing after three or
four iterations, with the half sum of squared residuals settling at its minimum; SciPy's `curve_fit` (which uses a
refined version of the same idea, Levenberg–Marquardt) agrees to all printed digits. The analytic Jacobian matched the
finite-difference one (difference $\sim10^{-9}$), so the method was fed the right derivatives. The remaining gap to the
truth is measurement noise. Why was the double-reciprocal estimate so poor? The next experiment answers that.

In [ ]:
# ── 7.2 Lineweaver–Burk vs Gauss–Newton over 200 repeated SIMULATED experiments ──
def gauss_newton_mm(S, v, n_iter=25):
    """Fit (Vmax, Km) by Gauss–Newton from the naive start (max v, median S); returns NaNs if it fails."""
    th = np.array([v.max(), np.median(S)])
    try:
        for _ in range(n_iter):
            r_ = th[0] * S / (th[1] + S) - v
            J_ = np.column_stack([S / (th[1] + S), -th[0] * S / (th[1] + S) ** 2])
            th = th + np.linalg.solve(J_.T @ J_, -J_.T @ r_)
    except np.linalg.LinAlgError:
        return np.full(2, np.nan)
    return th


rng_rep = np.random.default_rng(SEED + 7)
lb_est, gn_est = [], []
for _ in range(200):
    v_rep = MM_TRUE["Vmax"] * S_conc / (MM_TRUE["Km"] + S_conc) + rng_rep.normal(0, 0.3, S_conc.size)
    sl, ic = np.polyfit(1 / S_conc, 1 / v_rep, 1)
    lb_est.append([1 / ic, sl / ic]); gn_est.append(gauss_newton_mm(S_conc, v_rep))
lb_est, gn_est = np.array(lb_est), np.array(gn_est)
for name, est in [("Lineweaver–Burk", lb_est), ("Gauss–Newton", gn_est)]:
    q = np.nanpercentile(est[:, 0], [5, 50, 95])
    print(f"{name:16s}: Vmax median {q[1]:7.2f}, 90 % of estimates in [{q[0]:7.2f}, {q[2]:7.2f}]; "
          f"negative Vmax in {np.sum(est[:, 0] < 0)} of 200 runs; failures {np.isnan(est[:, 0]).sum()}")

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
Sg = np.linspace(0, 34, 300)
axes[0].plot(S_conc, v_obs, "ko", ms=4, label="SIMULATED data")
axes[0].plot(Sg, theta[0] * Sg / (theta[1] + Sg), color="tab:red", label="Gauss–Newton fit")
axes[0].plot(Sg, (1 / lb_icpt) * Sg / (lb_slope / lb_icpt + Sg), color="tab:orange", ls="--", label="Lineweaver–Burk")
axes[0].plot(Sg, 12 * Sg / (3 + Sg), "k:", label="truth")
axes[0].set_xlabel("S (mM)"); axes[0].set_ylabel("v (µmol/min/mg)"); axes[0].legend(fontsize=7); axes[0].set_title("fits")
axes[1].plot(1 / S_conc, 1 / v_obs, "ko", ms=4)
xr = np.linspace(0, 4.1, 50); axes[1].plot(xr, lb_icpt + lb_slope * xr, color="tab:orange", ls="--")
axes[1].set_xlabel("1/S"); axes[1].set_ylabel("1/v"); axes[1].set_title("double-reciprocal plot: low S dominates")
bins = np.linspace(0, 30, 61)
axes[2].hist(np.clip(lb_est[:, 0], 0, 30), bins=bins, alpha=0.6, color="tab:orange", label="Lineweaver–Burk")
axes[2].hist(gn_est[:, 0], bins=bins, alpha=0.7, color="tab:red", label="Gauss–Newton")
axes[2].axvline(12, color="k", ls=":"); axes[2].set_xlabel("estimated Vmax (clipped to 0–30)"); axes[2].legend(fontsize=7)
axes[2].set_title("200 repeated experiments"); plt.tight_layout(); plt.show()

🔍 **What just happened?** Over 200 simulated repetitions of the same experiment, Gauss–Newton estimates of $V_{max}$
cluster tightly around the truth 12 (median 11.98; 90 % between 11.62 and 12.32; no failures), while the Lineweaver–Burk
estimates scatter enormously (90 % between $-13.5$ and $36.4$) and in 17 of the 200 experiments $V_{max}$ even comes out
**negative**. The middle panel shows why: taking
reciprocals turns the smallest, noisiest velocities (at $S = 0.25$ mM, $v \approx 0.9$ with noise 0.3) into the largest values
on the $1/v$ axis, so a few unreliable points at the right end dictate the line. Transforming data to make a model
linear also transforms — and distorts — the noise. Fitting the original model by nonlinear least squares, with the
Jacobian supplying the local linearisation, avoids that.

✏️ **Your turn.** Fit the logistic growth curve $N(t) = K/(1 + (K/N_0 - 1)e^{-rt})$ of Module 14 to simulated OD data
with Gauss–Newton: write the three Jacobian columns ($\partial N/\partial N_0$, $\partial N/\partial K$, $\partial N/\partial r$)
with SymPy, check them with `numerical_jacobian`, and compare with `curve_fit`.

### 🌉 Bridge — curve fitting in the lab and back-propagation in deep learning

Gauss–Newton and its damped cousin **Levenberg–Marquardt** (inside `scipy.optimize.curve_fit` and `least_squares`, and
in GraphPad Prism's "nonlinear regression") fit most mechanistic curves of experimental biology: enzyme kinetics,
binding isotherms ($K_d$), dose–response (IC50, EC50), growth curves, decay of mRNA after transcriptional shut-off,
pharmacokinetic compartment models. Every one of them needs the Jacobian of the model with respect to its parameters.
In deep learning (BiologyToML
[22 · neural networks](../../../../ml/tutorial/BiologyToML/notebooks/22_neural_networks.ipynb)) the parameters number
in the millions, so instead of solving $(J^\top J)\Delta\boldsymbol\theta = -J^\top\mathbf r$ one takes cheap gradient steps
$-\eta J^\top\mathbf r$ (the gradient of $\frac12\lVert\mathbf r\rVert^2$), computed by back-propagation — the chain rule of
Section 6 as a product of layer Jacobians. Optimisation and its convexity are developed further in BiologyToML
[04 · calculus and optimisation](../../../../ml/tutorial/BiologyToML/notebooks/04_calculus_and_optimization.ipynb).

---
## 8 · Dynamical systems in several variables: the tRNA model

> 📍 **Where we are.** Module 14 solved one differential equation, $\frac{dy}{dt} = \alpha y$. Cells contain many
> interacting molecular species, so we need **systems** of equations — and the matrices of Modules 08–12 to handle
> them. Our running example comes from translation.

**Motivating example: tRNA molecules.** Assume all tRNA molecules of a cell are divided into two classes: **bound**
and **unbound** to an amino acid (charged and uncharged). Let the number of **unbound** tRNA molecules be $m_1$ and the
number of **bound** ones $m_2$. Four rates matter:

* $\alpha$ — rate of tRNA production (new tRNAs are made uncharged): molecules per unit time;
* $\beta$ — rate of amino acid binding (charging by aminoacyl-tRNA synthetases), per unbound molecule;
* $\gamma$ — rate of amino acid loss (unbinding; mostly by donating the amino acid to a growing protein on the
  ribosome), per bound molecule;
* $\delta$ — rate of tRNA degradation, per molecule of either class.

**Goal:** the change in the number of tRNA molecules in a cell over time.

**A warm-up with two independent variables.** If the two classes did not interact, each would follow its own Module 14
equation, $\frac{dm_1(t)}{dt} = r_1m_1(t)$ and $\frac{dm_2(t)}{dt} = r_2m_2(t)$, with solutions $m_i(t) = m_i(0)e^{r_it}$.
In general the classes **interact**, and the simplest interacting (linear) model is

$$\frac{dm_1(t)}{dt} = a\,m_1(t) + b\,m_2(t),\qquad \frac{dm_2(t)}{dt} = c\,m_1(t) + d\,m_2(t)
\qquad\text{or}\qquad \frac{d\mathbf m}{dt} = M\mathbf m,\quad M = \begin{bmatrix}a & b\\ c & d\end{bmatrix},$$

*in words:* each rate of change is a weighted sum of the current amounts; the matrix $M$ holds the weights (the warm-up
is the case of a diagonal $M$). Note that $\frac{d\mathbf m}{dt} = M\mathbf m$ is a linear map of Section 6 with Jacobian $M$.

**The tRNA equations.** Book-keeping of what enters and leaves each class per unit time:

$$
\begin{aligned}
\frac{dm_1}{dt} &= \alpha - \beta m_1 + \gamma m_2 - \delta m_1 = \alpha + (-\beta-\delta)\,m_1 + \gamma\,m_2\\
\frac{dm_2}{dt} &= \phantom{\alpha} + \beta m_1 - \gamma m_2 - \delta m_2 = \beta\,m_1 + (-\gamma-\delta)\,m_2
\end{aligned}
\qquad\Longleftrightarrow\qquad
\frac{d\mathbf m}{dt} = \underbrace{\begin{bmatrix}-\beta-\delta & \gamma\\ \beta & -\gamma-\delta\end{bmatrix}}_{M}\mathbf m + \underbrace{\begin{bmatrix}\alpha\\ 0\end{bmatrix}}_{\mathbf c} .
$$

*In words:* unbound tRNA is produced ($+\alpha$), lost to charging ($-\beta m_1$), regained when charged tRNA loses its amino
acid ($+\gamma m_2$) and degraded ($-\delta m_1$); bound tRNA gains what unbound loses to charging and loses unbinding and
degradation. A model of the form $\dot{\mathbf m} = M\mathbf m + \mathbf c$ is called **affine** (linear plus a constant).

**The discrete-time version.** If we only look once per time step (and the rates are fractions per step), the same
book-keeping says *new amount = old amount + change during the step*:

$$\mathbf m(t+1) = \mathbf m(t) + M\mathbf m(t) + \mathbf c = (I + M)\,\mathbf m(t) + \mathbf c ,$$

*in words:* the matrix of Module 08 that advances the state by one step is $I + M$, not $M$. (⚠️ Writing
$\mathbf m(t+1) = M\mathbf m(t) + \mathbf c$ with the $M$ above forgets that the molecules already present are still there.
And the derivative $\frac{d\mathbf m}{dt}$ is a *time derivative*, not a Jacobian — the Jacobian of the right-hand side with
respect to $\mathbf m$ is $M$.) For a step of length $\Delta t$ the rates scale: $\mathbf m(t + \Delta t) = (I + \Delta t M)\mathbf m(t) + \Delta t\,\mathbf c$ —
exactly Euler's method from Module 14.

In [ ]:
# ── 8.1 The tRNA model as a diagram, and its matrices (ILLUSTRATIVE rates; time unit ≈ minutes) ──
RATES = {"alpha": 50.0, "beta": 2.0, "gamma": 0.5, "delta": 0.1}
al, be, ga, de = RATES.values()
M_trna = np.array([[-be - de, ga], [be, -ga - de]])
c_trna = np.array([al, 0.0])
print("M =\n", M_trna, "\nc =", c_trna, "  det M =", round(np.linalg.det(M_trna), 4))

fig, ax = plt.subplots(figsize=(8, 3.3))
for xc_, lab, col in [(1.5, "m₁: unbound\n(uncharged) tRNA", "#cfe2f3"), (5.5, "m₂: bound\n(charged) tRNA", "#d9ead3")]:
    ax.add_patch(plt.Rectangle((xc_ - 1, 1), 2, 1.2, fc=col, ec="k", lw=1.5))
    ax.text(xc_, 1.6, lab, ha="center", va="center", fontsize=10)
arrow = dict(arrowstyle="-|>", lw=1.8, color="k", mutation_scale=15)
ax.annotate("", xy=(0.5, 1.6), xytext=(-0.8, 1.6), arrowprops=arrow); ax.text(-0.75, 1.75, "α (production)", fontsize=9)
ax.annotate("", xy=(4.5, 1.95), xytext=(2.5, 1.95), arrowprops=arrow); ax.text(3.5, 2.1, "β·m₁ (charging)", ha="center", fontsize=9)
ax.annotate("", xy=(2.5, 1.25), xytext=(4.5, 1.25), arrowprops=arrow); ax.text(3.5, 0.95, "γ·m₂ (unbinding)", ha="center", fontsize=9)
for xc_, lab in [(1.5, "δ·m₁"), (5.5, "δ·m₂")]:
    ax.annotate("", xy=(xc_, 0.2), xytext=(xc_, 1.0), arrowprops=arrow); ax.text(xc_ + 0.1, 0.45, lab + " (degradation)", fontsize=9)
ax.set_xlim(-1.2, 7.2); ax.set_ylim(0, 2.5); ax.axis("off"); ax.set_title("the tRNA model: flows between two pools")
plt.tight_layout(); plt.show()

🔍 **What just happened?** With the illustrative rates $\alpha = 50$ molecules per minute, $\beta = 2$, $\gamma = 0.5$ and
$\delta = 0.1$ per minute, the model matrix is $M = \begin{bmatrix}-2.1 & 0.5\\ 2 & -0.6\end{bmatrix}$ with $\det M = 0.26$,
and $\mathbf c = (50, 0)$. (Real tRNAs are much more stable than $\delta = 0.1$/min suggests — half-lives of hours or more; a
smaller $\delta$ only stretches the slow time-scale of Section 9.) The diagram is the book-keeping behind each matrix
entry: arrows leaving a pool give the negative diagonal entries, arrows between the pools the positive off-diagonal ones.

**Equilibria.** How do we find the levels at which nothing changes? An **equilibrium** (steady state) $\mathbf m^\ast$
satisfies $\frac{d\mathbf m}{dt} = \mathbf 0$:

$$M\mathbf m^\ast + \mathbf c = \mathbf 0 \quad\Longleftrightarrow\quad M\mathbf m^\ast = -\mathbf c \quad\Longleftrightarrow\quad \mathbf m^\ast = -M^{-1}\mathbf c \quad (\det M \ne 0).$$

*In words:* an equilibrium of an affine model is the solution of a linear system (Module 09). **Lemma:** a linear model
in continuous time has exactly one equilibrium, regardless of the number of variables, provided that the determinant of
$M$ is not 0 — because then $M$ is invertible and $M\mathbf m = -\mathbf c$ has exactly one solution (for $\mathbf c = \mathbf 0$ that
equilibrium is $\mathbf m^\ast = \mathbf 0$). For the tRNA model,
$\det M = (\beta + \delta)(\gamma + \delta) - \beta\gamma = \delta(\beta + \gamma + \delta)$, which is positive whenever tRNAs are
degraded at all ($\delta > 0$).

In [ ]:
# ── 8.2 The equilibrium symbolically (SymPy) and numerically; then simulate towards it ──
alpha_, beta_, gamma_, delta_ = sp.symbols("alpha beta gamma delta", positive=True)
M_sym = sp.Matrix([[-beta_ - delta_, gamma_], [beta_, -gamma_ - delta_]])
c_sym = sp.Matrix([alpha_, 0])
print("det M =", sp.factor(M_sym.det()))
m_star_sym = sp.simplify(-M_sym.inv() * c_sym)
print("m* = −M⁻¹c =", m_star_sym.T.tolist())
print("total m1* + m2* =", sp.simplify(sum(m_star_sym)), "| fraction charged m2*/(m1* + m2*) =",
      sp.simplify(m_star_sym[1] / sum(m_star_sym)))
m_star = -np.linalg.solve(M_trna, c_trna)
print(f"numbers: m* = {m_star.round(2)}, total {m_star.sum():.1f}, charged fraction {m_star[1] / m_star.sum():.4f}; "
      f"M m* + c = {(M_trna @ m_star + c_trna).round(10) + 0.0}")

trna_rhs = lambda t, m: M_trna @ m + c_trna
sol = integrate.solve_ivp(trna_rhs, (0, 60), [0.0, 0.0], t_eval=np.linspace(0, 60, 601), rtol=1e-9, atol=1e-9)
dt_ = 0.1
m_disc = [np.zeros(2)]
for _ in range(600):                                                 # discrete model with step Δt = 0.1: m ← (I + ΔtM)m + Δt c
    m_disc.append((np.eye(2) + dt_ * M_trna) @ m_disc[-1] + dt_ * c_trna)
m_disc = np.array(m_disc)
print(f"after 60 min from an empty pool: continuous {sol.y[:, -1].round(2)}, discrete (Δt = 0.1) {m_disc[-1].round(2)}")

fig, ax = plt.subplots(figsize=(6.4, 3.5))
ax.plot(sol.t, sol.y[0], color="tab:blue", label="m₁ unbound (continuous)")
ax.plot(sol.t, sol.y[1], color="tab:green", label="m₂ bound (continuous)")
ax.plot(sol.t[::20], m_disc[::20, 0], "o", ms=3, color="tab:blue", label="discrete steps, Δt = 0.1")
ax.plot(sol.t[::20], m_disc[::20, 1], "o", ms=3, color="tab:green")
for v_, col in zip(m_star, ["tab:blue", "tab:green"]):
    ax.axhline(v_, color=col, ls=":", lw=1)
ax.set_xlabel("time (min)"); ax.set_ylabel("tRNA molecules"); ax.legend(fontsize=7)
ax.set_title("ILLUSTRATIVE tRNA pools approach the equilibrium m* (dotted)"); plt.tight_layout(); plt.show()

🔍 **What just happened?** SymPy factors $\det M = \delta(\beta + \gamma + \delta)$ and solves for the equilibrium:

$$m_1^\ast = \frac{\alpha(\gamma + \delta)}{\delta(\beta + \gamma + \delta)}, \qquad m_2^\ast = \frac{\alpha\beta}{\delta(\beta + \gamma + \delta)},
\qquad m_1^\ast + m_2^\ast = \frac{\alpha}{\delta}, \qquad \frac{m_2^\ast}{m_1^\ast + m_2^\ast} = \frac{\beta}{\beta + \gamma + \delta}.$$

*In words:* the **total** tRNA pool settles where production balances degradation ($\alpha/\delta$ = 500 molecules here),
and the **charged fraction** is set by the competition between charging ($\beta$) and discharging plus loss
($\gamma + \delta$): $2/2.6 = 76.9\,\%$, i.e. $m^\ast = (115.38, 384.62)$, and $M\mathbf m^\ast + \mathbf c = \mathbf 0$ exactly. Starting from
an empty pool, the simulated continuous model and the discrete model with small steps ($\Delta t = 0.1$) both climb to that
equilibrium. Biologically, the formula predicts what happens in starvation: if an amino acid runs out, its synthetase
cannot charge ($\beta \to 0$) and the charged fraction collapses — in bacteria, uncharged tRNA on the ribosome is
precisely the signal that triggers the *stringent response*.

**What if $\det M = 0$?** Then $M$ is singular and $M\mathbf m = -\mathbf c$ has either **no** solution ($-\mathbf c$ is not in the
column space of $M$) or **infinitely many** (it is, and every vector of the null space can be added — Modules 09 and 11).
For the tRNA model this happens exactly when $\delta = 0$ (no degradation).

In [ ]:
# ── 8.3 det M = 0: no degradation (δ = 0). With production: no equilibrium. Without: a whole line of equilibria ──
M0 = M_sym.subs(delta_, 0)
print("δ = 0: M =", M0.tolist(), " det =", M0.det(), " null space spanned by", [list(v) for v in M0.nullspace()])
m1s, m2s = sp.symbols("m1 m2")
for label, alpha_val in [("α > 0 (production, no degradation)", alpha_), ("α = 0 (closed pool)", 0)]:
    eqs = M0 * sp.Matrix([m1s, m2s]) + sp.Matrix([alpha_val, 0])
    print(f"{label}: solve M m + c = 0 →", sp.solve(list(eqs), [m1s, m2s], dict=True))

M_nodeg = np.array([[-be, ga], [be, -ga]])
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for alpha_val, ax, title in [(50.0, axes[0], "α = 50, δ = 0: total grows for ever"), (0.0, axes[1], "α = 0, δ = 0: a line of equilibria")]:
    for start in ([0.0, 0.0], [300.0, 0.0], [100.0, 300.0], [0.0, 150.0]):
        s_ = integrate.solve_ivp(lambda t, m: M_nodeg @ m + np.array([alpha_val, 0.0]), (0, 20), start,
                                 t_eval=np.linspace(0, 20, 200))
        ax.plot(s_.y[0], s_.y[1], lw=1.5)
        ax.plot(*start, "o", ms=6, mfc="none", mec="k")                   # open circle = start
        ax.plot(*s_.y[:, -1], "k.", ms=8)                                 # dot = state after 20 min
    if alpha_val == 0:
        mm = np.linspace(0, 105, 10); ax.plot(mm, mm * be / ga, "k--", lw=1, label="β·m₁ = γ·m₂")
        ax.set_xlim(0, 320); ax.set_ylim(0, 420); ax.legend(fontsize=7)
    ax.set_xlabel("m₁ unbound"); ax.set_ylabel("m₂ bound"); ax.set_title(title, fontsize=9)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Without degradation $M = \begin{bmatrix}-\beta & \gamma\\ \beta & -\gamma\end{bmatrix}$ has determinant 0
(each column adds up to zero: charging and unbinding only move molecules between the pools, so they conserve the total). Its
null space is spanned by $(\gamma, \beta)$. With production ($\alpha > 0$), SymPy finds **no** equilibrium: the total grows by
$\alpha$ per minute for ever (left panel: trajectories run off towards the upper right). In a closed pool ($\alpha = 0$)
it finds **infinitely many**: every point of the line $\beta m_1 = \gamma m_2$ is an equilibrium, and each trajectory (right
panel) runs straight along the direction $(1, -1)$ — exchanging charged and uncharged tRNA at constant total — and stops on
that line at a point fixed by its own conserved total. So the lemma's condition $\det M \ne 0$ is exactly
what guarantees a single steady state.

✏️ **Your turn.** (1) Halve the charging rate $\beta$ (amino-acid shortage) and recompute $\mathbf m^\ast$: what happens to
the total and to the charged fraction? (2) Show by hand that for any $2\times2$ model whose columns each sum to $-\delta$,
$\frac{d}{dt}(m_1 + m_2) = \alpha - \delta(m_1 + m_2)$, and solve this one-variable equation with Module 14's method.

---
## 9 · Stability: towards or away from the equilibrium?

> 📍 **Where we are.** Section 8 found *where* the tRNA pools settle. But a cell is constantly disturbed — a burst of
> translation, a brief amino-acid shortage, cell division. In a given model, assuming we do not start at the
> equilibrium, do we move **towards** it or **away** from it? Section 1 asked the same question of a critical point of
> gradient descent. The answer comes from the **eigenvalues and eigenvectors** of $M$ (Module 12).

**Deviations obey the linear part.** Let $\mathbf u(t) = \mathbf m(t) - \mathbf m^\ast$ be the deviation from equilibrium.
Since $\mathbf m^\ast$ is constant and $M\mathbf m^\ast + \mathbf c = \mathbf 0$,

$$\frac{d\mathbf u}{dt} = \frac{d\mathbf m}{dt} = M\mathbf m + \mathbf c = M(\mathbf u + \mathbf m^\ast) + \mathbf c = M\mathbf u + \underbrace{(M\mathbf m^\ast + \mathbf c)}_{=\,\mathbf 0} = M\mathbf u .$$

*In words:* the constant inflow only shifts *where* the equilibrium is; how deviations evolve depends on $M$ alone.

**Assume we start on an eigenvector.** Suppose $\mathbf u(0) = \mathbf v$ with $M\mathbf v = \lambda\mathbf v$. Guess
$\mathbf u(t) = e^{\lambda t}\mathbf v$ and check by differentiating (Module 14):

$$\frac{d}{dt}\big(e^{\lambda t}\mathbf v\big) = \lambda e^{\lambda t}\mathbf v = e^{\lambda t}(\lambda\mathbf v) = e^{\lambda t}M\mathbf v = M\big(e^{\lambda t}\mathbf v\big) \;✓$$

*In words:* a deviation that starts along an eigenvector **stays on that line** and simply shrinks or grows like
$e^{\lambda t}$ — Module 14's $\frac{dy}{dt} = \alpha y$ with $\alpha = \lambda$. If $\lambda < 0$ we move towards the equilibrium,
if $\lambda > 0$ away from it. A general start is a combination of eigenvectors (they form a basis when they are
independent, Module 11), $\mathbf u(0) = c_1\mathbf v_1 + c_2\mathbf v_2$, and then

$$\mathbf u(t) = c_1e^{\lambda_1t}\mathbf v_1 + c_2e^{\lambda_2t}\mathbf v_2 .$$

So the equilibrium is **stable** (every deviation dies out) exactly when **all eigenvalues are negative** — or, if they
are complex, have negative real part; it is **unstable** if any eigenvalue has a positive real part. For the tRNA model
the eigen-pairs can be found by hand: $M\begin{pmatrix}\gamma\\ \beta\end{pmatrix} = -\delta\begin{pmatrix}\gamma\\ \beta\end{pmatrix}$
and $M\begin{pmatrix}1\\ -1\end{pmatrix} = -(\beta + \gamma + \delta)\begin{pmatrix}1\\ -1\end{pmatrix}$ (multiply it out!).

In [ ]:
# ── 9.1 Eigenvalues/eigenvectors of the tRNA matrix, and deviations that start on an eigenvector ──
for val, mult, vecs in M_sym.eigenvects():
    print(f"eigenvalue {sp.simplify(val)} (multiplicity {mult}) with eigenvector {[sp.simplify(e) for e in vecs[0]]}")
v_slow = np.array([ga, be]) / np.hypot(ga, be)                        # eigenvalue −δ
v_fast = np.array([1.0, -1.0]) / np.sqrt(2)                          # eigenvalue −(β + γ + δ)
lam_slow, lam_fast = -de, -(be + ga + de)
print(f"numbers: λ_slow = {lam_slow}, λ_fast = {lam_fast}; check M v = λ v:",
      np.allclose(M_trna @ v_slow, lam_slow * v_slow), np.allclose(M_trna @ v_fast, lam_fast * v_fast))
print("np.linalg.eigvals:", np.real_if_close(np.linalg.eigvals(M_trna)).round(6))

t_eval = np.linspace(0, 12, 241)
runs = {"start on v_slow": m_star + 150 * v_slow, "start on v_fast": m_star + 150 * v_fast,
        "generic start (0, 0)": np.zeros(2)}
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
for (name, start), col in zip(runs.items(), ["tab:purple", "tab:orange", "0.4"]):
    s_ = integrate.solve_ivp(trna_rhs, (0, 12), start, t_eval=t_eval, rtol=1e-10, atol=1e-10)
    dev = s_.y.T - m_star
    axes[0].semilogy(t_eval, np.linalg.norm(dev, axis=1), color=col, label=name)
    direction = dev[[0, 40, 120]] / np.linalg.norm(dev[[0, 40, 120]], axis=1)[:, None]
    print(f"{name:22s}: ‖u‖ at t = 0, 2, 6: {np.linalg.norm(dev[[0, 40, 120]], axis=1).round(3)}; "
          f"direction at t = 0, 2, 6: {direction.round(3).tolist()}")
axes[0].semilogy(t_eval, 150 * np.exp(lam_slow * t_eval), "k:", lw=1, label="150·e^(−0.1 t)")
axes[0].semilogy(t_eval, 150 * np.exp(lam_fast * t_eval), "k--", lw=1, label="150·e^(−2.6 t)")
axes[0].set_ylim(1e-6, 1e3); axes[0].set_xlabel("time (min)"); axes[0].set_ylabel("‖m(t) − m*‖ (log scale)")
axes[0].legend(fontsize=7); axes[0].set_title("deviations decay like e^(λt)")
plot_vectors([v_slow, 10 * M_trna @ v_slow, v_fast, M_trna @ v_fast],
             labels=["v_slow", "10·M·v_slow", "v_fast", "M·v_fast"],
             colors=["tab:purple", "tab:pink", "tab:orange", "tab:brown"], ax=axes[1], lim=(-2.9, 2.9),
             title="eigenvectors: M·v is a multiple of v")
plt.tight_layout(); plt.show()

🔍 **What just happened?** SymPy returns the eigenvalues $-\delta$ with eigenvector $(\gamma/\beta, 1) \propto (\gamma, \beta)$ and
$-(\beta + \gamma + \delta)$ with eigenvector $(-1, 1) \propto (1, -1)$ — for our rates $-0.1$ and $-2.6$, both negative, so the
equilibrium is **stable**. Started exactly on an eigenvector, the deviation keeps its direction (the printed unit
directions do not change: $(0.243, 0.970)$ for the slow one, $(0.707, -0.707)$ for the fast one) and its length follows
$150e^{\lambda t}$ precisely — straight lines on the log scale, with slopes $-0.1$ and $-2.6$. A generic start mixes both:
its deviation first drops fast (the fast mode dies within a couple of minutes) and then decays slowly along the slow
eigenvector. In the right panel $M\mathbf v$ is a negative multiple of $\mathbf v$ for both eigenvectors: $M\mathbf v_{fast} = -2.6\,\mathbf v_{fast}$, and $M\mathbf v_{slow} = -0.1\,\mathbf v_{slow}$ (drawn ten times longer, so it is
exactly $-\mathbf v_{slow}$).

**Biological reading.** The fast eigenvector $(1, -1)$ converts charged into uncharged tRNA without changing the total:
the **charging balance** re-equilibrates within about $1/2.6 \approx 0.4$ min. The slow eigenvector $(\gamma, \beta)$ keeps the two
pools in the proportion at which charging and unbinding exactly balance ($\beta m_1 = \gamma m_2$): it changes only the **total pool**,
which can only relax as fast as tRNAs are made and degraded ($1/\delta$ = 10 min here, hours in a real cell). One matrix, two time-scales — separated by the
eigenvalues.

In [ ]:
# ── 9.2 Phase portraits: the stable tRNA equilibrium vs a saddle (two genes that activate each other) ──
M_saddle = np.array([[-1.0, 2.0], [2.0, -1.0]])                      # linearised mutual activation, around (0, 0)
ev_s, V_s = np.linalg.eigh(M_saddle)                                # symmetric → real eigenvalues, ascending
print("saddle matrix eigenvalues:", ev_s, "eigenvectors (columns):", V_s.round(3).tolist())

fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
P1, P2 = np.meshgrid(np.linspace(0, 330, 25), np.linspace(180, 600, 25))
F = np.einsum("ij,jkl->ikl", M_trna, np.stack([P1, P2])) + c_trna[:, None, None]
axes[0].streamplot(P1, P2, F[0], F[1], color="0.75", density=1.0, linewidth=0.7)
for vec, col, lab in [(v_slow, "tab:purple", "slow eigenvector (λ = −0.1)"), (v_fast, "tab:orange", "fast eigenvector (λ = −2.6)")]:
    line = m_star[:, None] + np.outer(vec, np.linspace(-300, 300, 2))
    axes[0].plot(line[0], line[1], color=col, lw=5, alpha=0.55, label=lab, zorder=1)
for start in ([300, 200], [20, 590], [300, 590], [10, 200]):
    s_ = integrate.solve_ivp(trna_rhs, (0, 40), start, t_eval=np.linspace(0, 40, 400))
    axes[0].plot(s_.y[0], s_.y[1], color="k", lw=1.2, zorder=3); axes[0].plot(*start, "ko", ms=4)
axes[0].plot(*m_star, "r*", ms=14); axes[0].set_xlim(0, 330); axes[0].set_ylim(180, 600)
axes[0].set_xlabel("m₁ unbound"); axes[0].set_ylabel("m₂ bound"); axes[0].legend(fontsize=7, loc="upper right")
axes[0].set_title("tRNA model: stable node (both λ < 0)")
Q1, Q2 = np.meshgrid(np.linspace(-2, 2, 25), np.linspace(-2, 2, 25))
axes[1].streamplot(Q1, Q2, M_saddle[0, 0] * Q1 + M_saddle[0, 1] * Q2, M_saddle[1, 0] * Q1 + M_saddle[1, 1] * Q2,
                   color="0.75", density=1.0, linewidth=0.7)
for k, col in zip(range(2), ["tab:blue", "tab:red"]):
    axes[1].plot(*np.outer(V_s[:, k], [-3, 3]), color=col, lw=5, alpha=0.55, label=f"eigenvector, λ = {ev_s[k]:+.0f}",
                 zorder=1)
for start in ([1.8, -1.7], [-1.8, 1.75], [1.8, -1.9], [-1.8, 1.9], [0.3, -1.9], [-0.3, 1.9]):
    s_ = integrate.solve_ivp(lambda t, u: M_saddle @ u, (0, 3), start, t_eval=np.linspace(0, 3, 300))
    keep = np.all(np.abs(s_.y) < 2.05, axis=0)
    axes[1].plot(s_.y[0][keep], s_.y[1][keep], color="k", lw=1.2); axes[1].plot(*start, "ko", ms=4)
axes[1].plot(0, 0, "r*", ms=14); axes[1].set_xlim(-2, 2); axes[1].set_ylim(-2, 2); axes[1].set_aspect("equal")
axes[1].set_xlabel("u₁ (gene 1 deviation)"); axes[1].set_ylabel("u₂ (gene 2 deviation)"); axes[1].legend(fontsize=7)
axes[1].set_title("mutual activation: saddle (λ = +1 and −3)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Left: in the tRNA **phase portrait** (each point is a state $(m_1, m_2)$, the grey streamlines
show the direction of change) every trajectory runs into the red equilibrium. Trajectories first move almost parallel to
the orange fast eigenvector — the charging balance is restored within minutes — and then glide along the purple slow
eigenvector into the equilibrium: a **stable node**. Right: a linearised model of two genes that activate each other,
$M = \begin{bmatrix}-1 & 2\\ 2 & -1\end{bmatrix}$, has eigenvalues $+1$ (eigenvector $(1, 1)$, both genes up or both down) and $-3$
(eigenvector $(1, -1)$). Deviations along $(1, -1)$ die out, but any component along $(1, 1)$ grows like $e^{t}$: the
equilibrium is a **saddle**, unstable. Biologically this is the core of a **bistable switch** — the symmetric state is
abandoned and the cell commits to "both high" or "both low". (The synthetic genetic toggle switch of Gardner, Cantor
and Collins (2000) is built from two genes that *repress* each other; its symmetric state is a saddle in the same way, and the
cell commits to "gene 1 high, gene 2 low" or the reverse.)

**Discrete time.** For $\mathbf m(t+1) = (I + M)\mathbf m(t) + \mathbf c$ the deviations obey $\mathbf u(t+1) = (I + M)\mathbf u(t)$, and
starting on an eigenvector $\mathbf v$ of $M$ gives $\mathbf u(t) = (1 + \lambda)^t\mathbf v$ (Module 14's $\beta^x$). Stability
now requires $\lvert 1 + \lambda\rvert < 1$ for every eigenvalue $\lambda$ of $M$ — equivalently, all eigenvalues of $I + M$
inside the unit circle (Module 20 meets the same unit-circle idea in the speed at which a Markov chain forgets its start). With a time step $\Delta t$ the matrix is
$I + \Delta tM$ and the condition becomes $\lvert 1 + \Delta t\lambda\rvert < 1$: a perfectly stable continuous system can be
simulated into an *unstable* discrete one if the step is too long — the learning-rate limit of Section 1 in a new guise.

In [ ]:
# ── 9.3 Discrete-time tRNA model: stability requires |1 + Δt·λ| < 1 for both eigenvalues ──
fig, ax = plt.subplots(figsize=(6.4, 3.5))
for dt_step, col in [(0.1, "tab:blue"), (0.5, "tab:green"), (0.75, "tab:orange"), (0.8, "tab:red")]:
    step_matrix = np.eye(2) + dt_step * M_trna
    eig_step = np.real_if_close(np.linalg.eigvals(step_matrix))
    radius = np.abs(eig_step).max()
    m_ = np.array([300.0, 200.0]); traj = [m_]                         # start with too much uncharged tRNA
    for _ in range(int(round(30 / dt_step))):
        m_ = step_matrix @ m_ + dt_step * c_trna; traj.append(m_)
    traj = np.array(traj)
    print(f"Δt = {dt_step:4.2f}: eigenvalues of I + ΔtM = {np.sort(eig_step).round(3)}, largest |·| = {radius:.3f} "
          f"→ {'stable' if radius < 1 else 'UNSTABLE'}; m₁ after 30 min = {traj[-1, 0]:.1f}")
    ax.plot(np.arange(len(traj)) * dt_step, traj[:, 0], "-", lw=1.4 if dt_step < 0.8 else 1.0, color=col,
            label=f"Δt = {dt_step}", zorder=3 if dt_step < 0.8 else 2)
ax.axhline(m_star[0], color="k", ls=":", lw=1, label="m₁* (continuous equilibrium)")
ax.set_ylim(-400, 650); ax.set_xlabel("time (min)"); ax.set_ylabel("m₁ unbound"); ax.legend(fontsize=7, ncol=3, loc="lower center")
ax.set_title(f"step limit: Δt < 2/(β+γ+δ) = {2 / (be + ga + de):.3f} min"); plt.tight_layout(); plt.show()

🔍 **What just happened?** We started every run with a charging imbalance, $\mathbf m(0) = (300, 200)$. The eigenvalues of
$I + \Delta tM$ are $1 - 0.1\Delta t$ and $1 - 2.6\Delta t$. For $\Delta t = 0.1$ (0.99 and 0.74) and $\Delta t = 0.5$ (0.95 and
$-0.3$) both lie inside $(-1, 1)$ and the discrete model settles on the continuous equilibrium $m_1^\ast = 115.4$; at
$\Delta t = 0.75$ the fast eigenvalue is $-0.95$, so the fast mode flips sign every step and decays only slowly — the
orange zig-zag is still at 139.1 after 30 minutes; at $\Delta t = 0.8$ it is $-1.08$, outside the unit circle, and the
oscillation **grows** without bound ($m_1 = 3554$ after 30 minutes, after passing through impossible negative numbers of
molecules) although the biology is perfectly stable. The limit $\Delta t < 2/(\beta + \gamma + \delta) = 0.769$ is the same formula as the
learning-rate limit $\alpha < 2/f''$ of gradient descent and the Euler step limit of Module 14: three faces of one fact.

✏️ **Your turn.** (1) Change $\gamma$ to 3 (fast use of charged tRNA by very active ribosomes). Recompute the eigenvalues
and the charged fraction; which time-scale changed? (2) Construct a $2\times2$ matrix with complex eigenvalues
$-0.2 \pm i$ (e.g. $\begin{bmatrix}-0.2 & 1\\ -1 & -0.2\end{bmatrix}$), simulate $\dot{\mathbf u} = M\mathbf u$ and describe the
trajectories (a *stable spiral* — damped oscillations, as in many signalling pathways).

### 📘 Lecture notes — dynamical systems: the tRNA model, equilibria and stability

**tRNA model.** $m_1$ = unbound, $m_2$ = bound tRNA; $\alpha$ production, $\beta$ binding, $\gamma$ unbinding, $\delta$ degradation.
Continuous: $\frac{dm_1}{dt} = \alpha - (\beta + \delta)m_1 + \gamma m_2$, $\frac{dm_2}{dt} = \beta m_1 - (\gamma + \delta)m_2$, i.e.
$\frac{d\mathbf m}{dt} = M\mathbf m + \mathbf c$ with $M = \begin{bmatrix}-\beta-\delta & \gamma\\ \beta & -\gamma-\delta\end{bmatrix}$,
$\mathbf c = (\alpha, 0)^\top$. Discrete: $\mathbf m(t+1) = (I + M)\mathbf m(t) + \mathbf c$. Warm-ups: $\frac{dm_i}{dt} = r_im_i$ (uncoupled);
$\frac{d\mathbf m}{dt} = M\mathbf m$ with $M = \begin{bmatrix}a&b\\c&d\end{bmatrix}$ (coupled).

**Equilibria.** $M\mathbf m^\ast + \mathbf c = \mathbf 0 \Rightarrow \mathbf m^\ast = -M^{-1}\mathbf c$. **Lemma:** a linear model in continuous
time has exactly one equilibrium, regardless of the number of variables, provided $\det M \ne 0$. If $\det M = 0$: no
equilibrium or infinitely many (a line/plane — the null space). tRNA: $\det M = \delta(\beta + \gamma + \delta)$;
$m_1^\ast + m_2^\ast = \alpha/\delta$, charged fraction $\beta/(\beta + \gamma + \delta)$; $\delta = 0$ ⇒ $\det M = 0$.

**Stability.** Deviations obey $\dot{\mathbf u} = M\mathbf u$. **Start on an eigenvector** ($M\mathbf v = \lambda\mathbf v$): $\mathbf u(t) = e^{\lambda t}\mathbf v$
— towards the equilibrium if $\lambda < 0$, away if $\lambda > 0$. In general $\mathbf u(t) = \sum_k c_ke^{\lambda_kt}\mathbf v_k$: stable iff
all eigenvalues have negative real part (continuous time); iff all $\lvert 1 + \lambda\rvert < 1$, i.e. all eigenvalues of the
step matrix inside the unit circle (discrete time). tRNA: $\lambda = -\delta$ (eigenvector $(\gamma, \beta)$, total pool, slow) and
$\lambda = -(\beta + \gamma + \delta)$ (eigenvector $(1, -1)$, charging balance, fast). Mixed signs = saddle.

> 🎓 **Exam tip.** For a $2\times2$ system you can decide stability without computing eigenvalues: in continuous time
> both eigenvalues have negative real part iff $\operatorname{tr}M < 0$ and $\det M > 0$ (because
> $\operatorname{tr}M = \lambda_1 + \lambda_2$ and $\det M = \lambda_1\lambda_2$, Module 12). tRNA: trace $-(\beta + \gamma + 2\delta) < 0$,
> determinant $\delta(\beta + \gamma + \delta) > 0$ ⇒ stable.

---
## 10 · Nonlinear dynamics: linearising an epidemic with the Jacobian

> 📍 **Where we are.** Most biological dynamics are **nonlinear**: infection needs a susceptible *and* an infected
> person to meet (a product $SI$), enzymes saturate, genes activate each other through Hill functions. Sections 6 and 9
> combine into the standard method: find the equilibria, **linearise with the Jacobian** there, and read stability off
> its eigenvalues.

For $\frac{d\mathbf m}{dt} = F(\mathbf m)$ with an equilibrium $F(\mathbf m^\ast) = \mathbf 0$, local linearisation (Section 6) gives,
for the deviation $\mathbf u = \mathbf m - \mathbf m^\ast$,

$$\frac{d\mathbf u}{dt} = F(\mathbf m^\ast + \mathbf u) \approx \underbrace{F(\mathbf m^\ast)}_{=\,\mathbf 0} + J_F(\mathbf m^\ast)\,\mathbf u = J_F(\mathbf m^\ast)\,\mathbf u .$$

*In words:* close to an equilibrium every smooth system behaves like the linear system of Section 9 with
$M = J_F(\mathbf m^\ast)$, so the eigenvalues of the **Jacobian at the equilibrium** decide stability (the linear picture is
trustworthy as long as no eigenvalue has real part exactly 0). For a linear model $F(\mathbf m) = M\mathbf m + \mathbf c$ the
Jacobian is $M$ everywhere — which is why the tRNA analysis needed no linearisation.

**The SIR epidemic** (Kermack & McKendrick, 1927). Let $s, i, r$ be the fractions of a population that are susceptible,
infectious and recovered:

$$\frac{ds}{dt} = -\beta si, \qquad \frac{di}{dt} = \beta si - \gamma i, \qquad \frac{dr}{dt} = \gamma i .$$

*In words:* susceptibles become infected at a rate proportional to the contacts between the two groups ($\beta$ =
transmission rate), and infectious people recover at rate $\gamma$ (mean infectious period $1/\gamma$). ⚠️ The letters
$\beta, \gamma$ now mean transmission and recovery — not the tRNA rates. Since $s + i + r = 1$, the pair $(s, i)$ suffices.
Every state with $i = 0$ is an equilibrium (**disease-free**). At the disease-free state $(s, i) = (1, 0)$:

$$J = \begin{bmatrix}\dfrac{\partial\dot s}{\partial s} & \dfrac{\partial\dot s}{\partial i}\\[2mm] \dfrac{\partial\dot i}{\partial s} & \dfrac{\partial\dot i}{\partial i}\end{bmatrix}
= \begin{bmatrix}-\beta i & -\beta s\\ \beta i & \beta s - \gamma\end{bmatrix} \;\xrightarrow{(s,\,i)\,=\,(1,\,0)}\; \begin{bmatrix}0 & -\beta\\ 0 & \beta - \gamma\end{bmatrix} .$$

*In words:* the matrix is upper triangular, so its eigenvalues are the diagonal entries (Module 12): $0$ (moving along
the line of disease-free states changes nothing — the $\det = 0$ situation of Section 8) and $\beta - \gamma$. A few
introduced infections therefore grow like $e^{(\beta - \gamma)t}$ if $\beta > \gamma$ and die out if $\beta < \gamma$:

$$\text{an epidemic can start} \iff R_0 = \frac{\beta}{\gamma} > 1 ,$$

the **basic reproduction number** — the expected number of people one case infects in a fully susceptible population.
If a fraction $p$ is immune (vaccinated), the disease-free state is $(1 - p, 0)$ and the eigenvalue becomes
$\beta(1 - p) - \gamma$, negative when $p > 1 - 1/R_0$: the **herd-immunity threshold**.

In [ ]:
# ── 10.1 The SIR Jacobian symbolically, its eigenvalues at disease-free states, and simulations ──
s_s, i_s, b_s, g_s, p_s = sp.symbols("s i beta gamma p", positive=True)
F_sir = sp.Matrix([-b_s * s_s * i_s, b_s * s_s * i_s - g_s * i_s])
J_sir = F_sir.jacobian(sp.Matrix([s_s, i_s]))
print("J =", J_sir.tolist())
print("at (1, 0): J =", J_sir.subs({s_s: 1, i_s: 0}).tolist(), "eigenvalues", list(J_sir.subs({s_s: 1, i_s: 0}).eigenvals()))
print("at (1 − p, 0): eigenvalues", list(J_sir.subs({s_s: 1 - p_s, i_s: 0}).eigenvals()))


def sir_rhs(t, y, beta, gamma):
    """SIR model for fractions (s, i, r)."""
    s, i, r = y
    return [-beta * s * i, beta * s * i - gamma * i, gamma * i]


GAMMA = 0.2                                                           # recovery rate per day (5-day infectious period)
scenarios = {"R0 = 2.5": (0.5, 0.0), "R0 = 0.8": (0.16, 0.0), "R0 = 2.5, 65 % vaccinated": (0.5, 0.65)}
days = np.arange(0, 161)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.7))
for (name, (beta, p_vac)), col in zip(scenarios.items(), ["tab:red", "tab:blue", "tab:green"]):
    y0 = [1 - p_vac - 1e-4, 1e-4, p_vac]
    sol_ = integrate.solve_ivp(sir_rhs, (0, 160), y0, args=(beta, GAMMA), t_eval=days, rtol=1e-9, atol=1e-12)
    lam_lin = beta * (1 - p_vac) - GAMMA                                # eigenvalue of the linearisation
    early = np.polyfit(days[:20], np.log(sol_.y[1][:20]), 1)[0]        # observed early growth rate of log i
    print(f"{name:26s}: predicted rate β(1−p) − γ = {lam_lin:+.3f}/day, observed {early:+.3f}/day; "
          f"peak i = {sol_.y[1].max():.4f} on day {days[sol_.y[1].argmax()]}; finally infected {sol_.y[2][-1] - p_vac:.3f}")
    axes[0].semilogy(days, sol_.y[1], color=col, label=name)
    axes[0].semilogy(days[:60], 1e-4 * np.exp(lam_lin * days[:60]), color=col, ls=":", lw=1)
    if name == "R0 = 2.5":
        for k, (lab, c_) in enumerate(zip("SIR", ["tab:blue", "tab:red", "tab:green"])):
            axes[1].plot(days, sol_.y[k], color=c_, label=lab)
axes[0].set_ylim(1e-7, 1); axes[0].set_xlabel("day"); axes[0].set_ylabel("infectious fraction i (log)")
axes[0].legend(fontsize=7); axes[0].set_title("dotted: linearisation e^((β(1−p) − γ)t)")
axes[1].set_xlabel("day"); axes[1].set_ylabel("fraction of population"); axes[1].legend()
axes[1].set_title("R0 = 2.5 (β = 0.5, γ = 0.2 per day; ILLUSTRATIVE)")
plt.tight_layout(); plt.show()
print(f"herd-immunity threshold 1 − 1/R0 for R0 = 2.5: {1 - 1 / 2.5:.0%}")

🔍 **What just happened?** SymPy's Jacobian at $(1, 0)$ is $\begin{bmatrix}0 & -\beta\\ 0 & \beta - \gamma\end{bmatrix}$ with eigenvalues 0
and $\beta - \gamma$, and at $(1 - p, 0)$ the second eigenvalue is $\beta(1 - p) - \gamma$. The simulations confirm the
linearisation where it should hold — at the start, while $i$ is tiny: with $R_0 = 2.5$ the infectious fraction first grows
at 0.297 per day, almost exactly the predicted $\beta - \gamma = 0.3$ (a doubling time of $\ln 2/0.3 \approx 2.3$ days), peaks at
23 % of the population around day 32 and ends with 89 % ever infected; with $R_0 = 0.8$ it declines at the predicted rate
$-0.04$ per day; and vaccinating 65 % — above the herd-immunity threshold of 60 % — turns the eigenvalue negative
($-0.025$) so the same pathogen fizzles out. Far from the equilibrium (the peak and the decline) the nonlinear terms take
over, which is why the dotted exponential lines eventually part from the curves.

✏️ **Your turn.** (1) For a pathogen with infectious period 7 days and $R_0 = 4$, compute $\beta$, $\gamma$, the early growth
rate and the herd-immunity threshold; check by simulation. (2) Add waning immunity ($\frac{dr}{dt} = \gamma i - \omega r$,
$\frac{ds}{dt} = -\beta si + \omega r$) and find the **endemic** equilibrium with SymPy; compute the Jacobian there — are its
eigenvalues real or complex?

### 🌉 Bridge — calculus across biology and machine learning

**Applications of calculus in biology** (this module and Module 14): modelling the spread of diseases with differential
equations (SIR, $R_0$, herd immunity — Section 10); analysing reaction rates in metabolic networks (Michaelis–Menten,
mass-action models, steady states $S\mathbf v = \mathbf 0$ of Module 09); modelling tumour growth and treatment response
(Gompertz and logistic models, fitted by Gauss–Newton); population dynamics and pharmacokinetics (linear compartment
models are exactly $\dot{\mathbf m} = M\mathbf m + \mathbf c$, and their eigenvalues are the observed elimination phases); and
parameter optimisation in biological machine-learning models. Cell-state dynamics — RNA velocity and fate decisions
with their stable states and saddles — are the subject of BiologyToML
[32](../../../../ml/tutorial/BiologyToML/notebooks/32_cell_dynamics_velocity.ipynb).

**Why calculus matters in machine learning:** loss functions are minimised via derivatives (gradients, Sections 1–3);
back-propagation in neural networks uses the chain rule (Section 6; BiologyToML
[22](../../../../ml/tutorial/BiologyToML/notebooks/22_neural_networks.ipynb)); Jacobians and Hessians are used in complex
optimisation and modelling (Sections 3.5, 6, 7); and gradient descent is learning through calculus. Maximum likelihood
(Module 23) and regression (Module 27) are the next places you will set gradients to zero.

## 📌 Key takeaways

- **Gradient descent** $x \leftarrow x - \alpha f'(x)$ follows from $f(x+\varepsilon) \approx f(x) + \varepsilon f'(x)$: step against the
  slope. A critical point can be **stable** ($x^2/2$: nearby starts are drawn in) or **unstable** ($-x^2/2$); the
  learning rate must satisfy $\alpha < 2/f''$; descent finds *a* local minimum, not necessarily the global one (Cauchy, 1847).
- For $f:\mathbb R^n\to\mathbb R$, **partial derivatives** vary one input at a time; the **gradient** collects them (dimension
  = number of inputs), points in the direction of steepest ascent and is perpendicular to contour lines. Examples:
  $\nabla(\mathbf x^\top\mathbf x) = 2\mathbf x$, $\nabla\lVert\mathbf x\rVert = \mathbf x/\lVert\mathbf x\rVert$.
- **Neural networks learn** by $\mathbf v \leftarrow \mathbf v - \eta\nabla c$, because $\Delta c \approx \nabla c\cdot\Delta\mathbf v = -\eta\lVert\nabla c\rVert^2$;
  the step limit is $2/\lambda_{max}$ of the **Hessian**, whose eigenvalue signs also classify critical points (min, max, saddle).
- **Identities:** $\nabla(\mathbf b^\top\mathbf x) = \mathbf b$, $\nabla(\mathbf x^\top A\mathbf x) = (A + A^\top)\mathbf x = 2A\mathbf x$ for symmetric $A$.
- **Least squares:** when $\mathbf b \notin \operatorname{span}(\operatorname{col}A)$, minimise $\lVert A\mathbf x - \mathbf b\rVert^2 = \mathbf x^\top A^\top A\mathbf x - 2\mathbf x^\top A^\top\mathbf b + \mathbf b^\top\mathbf b$;
  gradient $2A^\top A\mathbf x - 2A^\top\mathbf b$; **normal equations** $A^\top A\mathbf x = A^\top\mathbf b$; residual ⟂ columns. Standardise
  predictors before gradient descent.
- The **Jacobian** ($m\times n$, rows = outputs) gives local linearisation $g(\mathbf x + d\mathbf x) \approx g(\mathbf x) + J\,d\mathbf x$,
  the chain rule $J_{h\circ g} = J_hJ_g$ (back-propagation), and **Gauss–Newton** $(J^\top J)\Delta\boldsymbol\theta = -J^\top\mathbf r$ for
  nonlinear fits — far better than linearising the data (Lineweaver–Burk).
- **Linear dynamics** $\dot{\mathbf m} = M\mathbf m + \mathbf c$ (tRNA: bound/unbound): equilibrium $\mathbf m^\ast = -M^{-1}\mathbf c$, unique iff
  $\det M \ne 0$ (else none or infinitely many). **Start on an eigenvector**: deviations evolve as $e^{\lambda t}\mathbf v$; stable
  iff all eigenvalues have negative real part (discrete time: $\lvert 1 + \lambda\rvert < 1$).
- **Nonlinear dynamics:** linearise with the Jacobian at the equilibrium. SIR: eigenvalue $\beta - \gamma$ at the disease-free
  state ⇒ outbreak iff $R_0 = \beta/\gamma > 1$; herd immunity at $1 - 1/R_0$.

---
## ✅ Checkpoint

1. Gradient descent on $f(x) = 2x^2$ uses $x_{t+1} = x_t - \alpha f'(x_t)$. For which $\alpha$ does it converge, and for which does it oscillate?
   <details><summary>Answer</summary>$f'(x) = 4x$, so $x_{t+1} = (1 - 4\alpha)x_t$. Convergence iff $\lvert 1 - 4\alpha\rvert < 1$, i.e. $0 < \alpha < 0.5$ ($= 2/f''$); oscillating (sign flips) for $0.25 < \alpha < 0.5$; one-step convergence at $\alpha = 0.25$; divergence for $\alpha > 0.5$.</details>
2. Gradient descent on $g(x) = -x^2/2$ is started at $x_0 = 0.01$. What happens, and what does it say about the critical point $x = 0$?
   <details><summary>Answer</summary>$x_{t+1} = x_t - \alpha(-x_t) = (1 + \alpha)x_t$: the iterate grows geometrically away from 0. The critical point 0 is **unstable** for $g$ (it is a maximum); for $f = x^2/2$ the same rule gives $(1-\alpha)x_t \to 0$ (stable).</details>
3. Compute the gradient of $f(x_1, x_2) = x_1^2x_2 + 3x_2$ and evaluate it at $(1, 2)$. In which direction should you move to decrease $f$ fastest?
   <details><summary>Answer</summary>$\nabla f = (2x_1x_2,\; x_1^2 + 3) = (4, 4)$ at $(1, 2)$. Fastest decrease along $-\nabla f$, i.e. the direction $(-1, -1)/\sqrt2$.</details>
4. Show that $\Delta\mathbf v = -\eta\nabla c$ decreases the cost (for small $\eta$), starting from the linear approximation.
   <details><summary>Answer</summary>$\Delta c \approx \nabla c\cdot\Delta\mathbf v = \nabla c\cdot(-\eta\nabla c) = -\eta\lVert\nabla c\rVert^2 \le 0$, with equality only if $\nabla c = \mathbf 0$ (a critical point). For larger $\eta$ the neglected second-order terms can make $c$ increase.</details>
5. For $A = \begin{bmatrix}1&2\\0&3\end{bmatrix}$ and $\mathbf x = (1, 1)$, compute $\nabla(\mathbf x^\top A\mathbf x)$. Why is $2A\mathbf x$ wrong here?
   <details><summary>Answer</summary>$(A + A^\top)\mathbf x = \begin{bmatrix}2&2\\2&6\end{bmatrix}(1,1)^\top = (4, 8)$. $2A\mathbf x = (6, 6)$ is wrong because $A$ is not symmetric; check: $\mathbf x^\top A\mathbf x = x_1^2 + 2x_1x_2 + 3x_2^2$ has partials $2x_1 + 2x_2 = 4$ and $2x_1 + 6x_2 = 8$.</details>
6. Fit $y = x_1 + x_2t$ to the points $(t, y) = (1, 1), (2, 2), (3, 2)$ by least squares, and verify that the residual is orthogonal to the columns.
   <details><summary>Answer</summary>$A = \begin{bmatrix}1&1\\1&2\\1&3\end{bmatrix}$, $\mathbf b = (1,2,2)$: $A^\top A = \begin{bmatrix}3&6\\6&14\end{bmatrix}$, $A^\top\mathbf b = (5, 11)$, $\det = 6$, $\mathbf x = \frac16\begin{bmatrix}14&-6\\-6&3\end{bmatrix}(5,11)^\top = (2/3, 1/2)$. Residual $\mathbf b - A\mathbf x = (-1/6, 1/3, -1/6)$; $A^\top\mathbf r = (0, -1/6 + 2/3 - 1/2) = (0, 0)$ ✓.</details>
7. In the expansion of $\lVert A\mathbf x - \mathbf b\rVert^2$, why do $-\mathbf x^\top A^\top\mathbf b$ and $-\mathbf b^\top A\mathbf x$ combine into $-2\mathbf x^\top A^\top\mathbf b$, and what is wrong with writing the middle term as $-2A\mathbf x\mathbf b^\top$?
   <details><summary>Answer</summary>$\mathbf b^\top A\mathbf x$ is a $1\times1$ number, and a number equals its transpose: $(\mathbf b^\top A\mathbf x)^\top = \mathbf x^\top A^\top\mathbf b$. $A\mathbf x\mathbf b^\top$ is an $m\times m$ matrix (column times row), so it cannot be a term in a sum of numbers.</details>
8. Compute the Jacobian of $g(\mathbf x) = (x_1x_2,\; x_1 + x_2^2)$ at $(1, 2)$, and use it to predict $g(1.01, 1.98)$.
   <details><summary>Answer</summary>$J_g = \begin{bmatrix}x_2 & x_1\\ 1 & 2x_2\end{bmatrix} = \begin{bmatrix}2&1\\1&4\end{bmatrix}$; $g(1,2) = (2, 5)$; $d\mathbf x = (0.01, -0.02)$, $J\,d\mathbf x = (0, -0.07)$, so $g \approx (2, 4.93)$. Exact: $(1.9998, 4.9304)$.</details>
9. Why does Gauss–Newton solve $(J^\top J)\Delta\boldsymbol\theta = -J^\top\mathbf r$, and why is fitting Michaelis–Menten through a Lineweaver–Burk plot risky?
   <details><summary>Answer</summary>Linearising $\mathbf r(\boldsymbol\theta + \Delta\boldsymbol\theta) \approx \mathbf r + J\Delta\boldsymbol\theta$ and minimising $\lVert\mathbf r + J\Delta\boldsymbol\theta\rVert^2$ is linear least squares with $A = J$, $\mathbf b = -\mathbf r$, whose normal equations are $J^\top J\Delta\boldsymbol\theta = -J^\top\mathbf r$. Taking reciprocals turns the smallest, noisiest velocities into the largest values of $1/v$, which then dominate the straight-line fit (Section 7.2: wildly variable, sometimes negative $V_{max}$).</details>
10. tRNA model with $\alpha = 20$, $\beta = 1$, $\gamma = 1$, $\delta = 0.1$: find the total pool, the charged fraction and $\mathbf m^\ast$, the eigenvalues, and decide stability. What changes if $\delta = 0$?
    <details><summary>Answer</summary>Total $\alpha/\delta = 200$; charged fraction $\beta/(\beta+\gamma+\delta) = 1/2.1 = 0.476$; $\mathbf m^\ast = (104.8, 95.2)$. Eigenvalues $-\delta = -0.1$ and $-(\beta+\gamma+\delta) = -2.1$: both negative, stable. With $\delta = 0$, $\det M = 0$: with $\alpha > 0$ there is no equilibrium (the pool grows by $\alpha$ per unit time); with $\alpha = 0$ every point of the line $\beta m_1 = \gamma m_2$ is an equilibrium.</details>
11. The discrete tRNA model $\mathbf m(t+1) = (I + M)\mathbf m(t) + \mathbf c$ is run with the per-step rates $\beta = 2$, $\gamma = 0.5$, $\delta = 0.1$. Is it stable?
    <details><summary>Answer</summary>The eigenvalues of $I + M$ are $1 - \delta = 0.9$ and $1 - (\beta + \gamma + \delta) = -1.6$. Since $\lvert -1.6\rvert > 1$ the discrete model oscillates with growing amplitude: unstable (and meaningless — more than 100 % of the unbound pool would be charged per step). The continuous model with the same rates is stable.</details>
12. An infection has $\beta = 0.3$ and $\gamma = 0.1$ per day. Compute $R_0$, the early growth rate and doubling time of cases, and the herd-immunity threshold, using the Jacobian at the disease-free state.
    <details><summary>Answer</summary>$R_0 = 3$; the Jacobian's non-zero eigenvalue at $(1, 0)$ is $\beta - \gamma = 0.2$ per day, so cases double every $\ln 2/0.2 \approx 3.5$ days; herd immunity at $1 - 1/R_0 = 2/3 \approx 67\,\%$.</details>

## 🧠 Questions & discussion

1. A neural network with millions of parameters is trained by gradient descent with a single learning rate. Why might one rate be too large for some directions and too small for others, and what does that have to do with the eigenvalues of the Hessian? How do "adaptive" optimisers try to fix this?
2. Lineweaver–Burk plots were standard in biochemistry for decades because a straight line could be fitted with a ruler. Which other transformations used in biology (log of counts, Scatchard plots, log-linear growth fits) change the noise structure, and when is that acceptable?
3. The tRNA model assumes that charging is proportional to the number of unbound tRNAs ($\beta m_1$). Real synthetases saturate (Michaelis–Menten) and depend on amino-acid supply. How would you change the model, and how would you analyse its stability now? (Hint: Section 10.)
4. The SIR model assumes that everybody meets everybody with equal probability. Which biological and social facts violate this assumption, and how could they change the threshold $R_0 > 1$?
5. A stable equilibrium can be approached monotonically (real negative eigenvalues) or with damped oscillations (complex eigenvalues with negative real part). Which biological systems oscillate in this way after a perturbation (think of hormones, the p53–Mdm2 feedback loop, circadian clocks), and what would that imply about their Jacobians?

## 🚀 Bonus — going further

- **Newton's method in many variables.** Replace the learning rate by the inverse Hessian: $\mathbf v \leftarrow \mathbf v - H^{-1}\nabla c$. Implement it for the one-neuron dose–response model (compute $H$ with `sp.hessian` or by finite differences of the gradient) and count the iterations versus gradient descent.
- **Stochastic gradient descent.** Estimate the gradient of the insulin cost from a random mini-batch of 10 people per step instead of all 120. Plot the noisy cost curve and compare with full-batch descent; this is how neural networks are trained on large data sets.
- **Least squares without normal equations.** Solve the insulin regression with a QR decomposition (`np.linalg.qr`): $A = QR$ gives $R\mathbf x = Q^\top\mathbf b$. Compare the accuracy with the normal equations when two predictors are nearly collinear (condition number of $A^\top A$ = condition number of $A$ squared).
- **Oscillating ecosystems.** Linearise the Lotka–Volterra predator–prey model $\dot x = ax - bxy$, $\dot y = -cy + dxy$ at its coexistence equilibrium $(c/d, a/b)$: the Jacobian has purely imaginary eigenvalues $\pm i\sqrt{ac}$, and the populations cycle. What does the linearisation predict for the period?
- **A genetic toggle switch.** Model two mutually repressing genes, $\dot u = \frac{a}{1 + v^n} - u$, $\dot v = \frac{a}{1 + u^n} - v$; find the equilibria numerically for $a = 3$, $n = 2$, compute the Jacobian at each and classify them (two stable nodes and a saddle) — the design of Gardner, Cantor & Collins (2000).

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module:            15 · Gradients, Jacobians, optimization and dynamics
Input:             toy functions (±x²/2, a quartic, bowls); SIMULATED insulin study (simulate_insulin_study);
                   SIMULATED enzyme velocities (Vmax 12, Km 3 mM) and dose–response wells (IC50 2 µM);
                   ILLUSTRATIVE tRNA rates (α 50, β 2, γ 0.5, δ 0.1 per min) and SIR rates (β 0.5, γ 0.2 per day)
Output:            gradient-descent paths and step-size limits; gradients and Jacobians (checked numerically);
                   least-squares coefficients; Gauss–Newton fits; equilibria, eigenvalues, phase portraits, R0
Method:            gradient descent; partial derivatives and finite differences; normal equations / lstsq;
                   Gauss–Newton; −M⁻¹c for equilibria; eigen-decomposition for stability; Jacobian linearisation
Key equation:      v ← v − η∇c ;  AᵀA x = Aᵀb ;  g(x + dx) ≈ g(x) + J dx ;  du/dt = J(m*) u ⇒ u = e^{λt} v
Key parameters:    learning rate η (vs 2/λmax), standardisation, number of Gauss–Newton iterations, time step Δt
What changed when I modified …  (e.g. η = 0.22 → 0.25; raw vs standardised predictors; Δt = 0.75 → 0.8; δ → 0)
One failure mode:  too large a step diverges; gradient descent stops in a local minimum; det M = 0 has no unique equilibrium
Connection to 02-680: gradient-based optimisation, gradients, least squares, Jacobians, the tRNA model,
                   equilibria (det M lemma) and stability via eigenvectors
```

## 📚 References

- Deisenroth, M. P., Faisal, A. A. & Ong, C. S. (2020) *Mathematics for Machine Learning*. Cambridge University Press — ch. 5 (vector calculus: partial derivatives, gradients, Jacobians, gradients of matrices, back-propagation, useful identities) and ch. 7 (gradient descent).
- Strang, G. *Introduction to Linear Algebra*. Wellesley-Cambridge Press — least squares and projections; eigenvalues and differential equations $d\mathbf u/dt = A\mathbf u$.
- Betti M. *Mathematics and Statistics for the Quantitative Sciences* — systems of differential equations, equilibria and stability in biological models.
- Isaak, S. & Manougian, M. N. *Basic Concepts of Linear Algebra* — systems of equations, determinants and eigenvalues used in Sections 5 and 8–9.
- D. DeBlasio, 02-680 lecture notes, github.com/deblasiolab/02-680_Documents.
- Cauchy, A. (1847) Méthode générale pour la résolution des systèmes d'équations simultanées. *Comptes rendus de l'Académie des Sciences* 25:536–538.
- Rumelhart, D. E., Hinton, G. E. & Williams, R. J. (1986) Learning representations by back-propagating errors. *Nature* 323:533–536.
- Lineweaver, H. & Burk, D. (1934) The determination of enzyme dissociation constants. *Journal of the American Chemical Society* 56:658–666.
- Levenberg, K. (1944) A method for the solution of certain non-linear problems in least squares. *Quarterly of Applied Mathematics* 2:164–168; Marquardt, D. W. (1963) An algorithm for least-squares estimation of nonlinear parameters. *Journal of the Society for Industrial and Applied Mathematics* 11:431–441.
- Kermack, W. O. & McKendrick, A. G. (1927) A contribution to the mathematical theory of epidemics. *Proceedings of the Royal Society of London A* 115:700–721.
- Gardner, T. S., Cantor, C. R. & Collins, J. J. (2000) Construction of a genetic toggle switch in *Escherichia coli*. *Nature* 403:339–342.

## ➡️ Next

**[Module 16 · 🎲 Probability: sample spaces, events and axioms](16_probability_spaces.ipynb)** — with Part D complete you can describe how biological quantities **change** (derivatives,
differential equations, stability) and how to **fit** models by optimisation. Part E turns from change to **chance**.
Measurements are noisy, genotypes are inherited at random, and the residuals of every least-squares fit in this module
were treated as unexplained error. Module 16 builds the language for that uncertainty — sample spaces, events and the
axioms of probability — and later modules come back to this one: maximum likelihood (Module 23) sets the gradient of a
log-likelihood to zero, and Module 27 shows that least squares *is* maximum likelihood under Gaussian noise.